In [0]:
# DBTITLE 1,Vibe Modelling Agent - Comprehensive Tester
# Databricks notebook source

import json
import os
import re
import random
import time
import uuid
from datetime import datetime
from pyspark.sql import SparkSession

# ═══════════════════════════════════════════════════════════════════
# WIDGETS — only thing executed at module level besides definitions
# ═══════════════════════════════════════════════════════════════════

dbutils.widgets.text("business_name", "", "01. Business Name")
dbutils.widgets.text("business_description", "", "02. Business Description")
dbutils.widgets.text("model_vibes", "", "03. Vibes (for vibe modeling test)")
dbutils.widgets.text("catalog", "", "04. Catalog (metamodel + deployment)")


# ═══════════════════════════════════════════════════════════════════
# PARAMETER POOLS — every dropdown option from the agent notebook
# ═══════════════════════════════════════════════════════════════════

PARAM_POOLS = {
    "naming_convention": ["snake_case", "camelCase", "PascalCase", "SCREAMING_CASE"],
    "table_id_type": ["BIGINT", "INT", "LONG", "STRING"],
    "boolean_format": ["Boolean (True/False)", "Int (0/1)", "String (Y/N)"],
    "date_format": ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "yyyy/MM/dd", "dd-MM-yyyy"],
    "timestamp_format": [
        "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
        "yyyy-MM-dd HH:mm:ss",
        "yyyy-MM-dd'T'HH:mm:ss",
        "yyyy-MM-dd HH:mm:ss.SSS",
    ],
    "housekeeping_columns": ["No", "Yes"],
    "history_tracking_columns": ["No", "Yes"],
    "cataloging_style": ["One Catalog", "Catalog per Division", "Catalog per Domain"],
    "org_divisions_ecm": ["Operations, Business and Corporate"],
    "org_divisions_mvm": ["Operations", "Operations and Business"],
    "primary_key_suffix": ["_id", "_key", "_pk", "Id", "_identifier"],
    "schema_prefix": ["stg_", "raw_", "src_", "dw_"],
    "schema_suffix": ["", "_layer", "_zone"],
    "tag_prefix": ["dbx_", "tag_", "meta_", ""],
    "tag_suffix": ["", "_v1", "_tag"],
    "catalog_prefix": ["cat_", "uc_", "db_"],
    "catalog_suffix": ["", "_zone", "_catalog"],
    "classification_levels": [
        "restricted=restricted, confidential=confidential, internal=Internal, public=public",
        "top_secret=Top Secret, secret=Secret, classified=Classified, unclassified=Unclassified",
        "high=High Sensitivity, medium=Medium Sensitivity, low=Low Sensitivity",
    ],
    "generate_samples_nonzero": ["3"],
}


# ═══════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — pure definitions, no execution
# ═══════════════════════════════════════════════════════════════════

def _sanitize_name_for_runner(name):
    """Match the runner's sanitize_name(name, strip_stop_words=False) exactly."""
    if not name:
        return "unnamed_model"
    s = str(name).lower()
    s = re.sub(r'[^a-z0-9_]', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    if s and s[0].isdigit():
        s = '_' + s
    if not s or s == '_':
        first_word = re.sub(r'[^a-z0-9]', '', str(name).lower().split()[0] if str(name).split() else '')
        return first_word if first_word else "unnamed_model"
    return s


def _pick(pool_key):
    return random.choice(PARAM_POOLS[pool_key])


def _random_conventions(test_id=""):  # test_id kept for backward compat but unused
    return {
        "naming_convention": _pick("naming_convention"),
        "primary_key_suffix": _pick("primary_key_suffix"),
        "schema_prefix": _pick("schema_prefix"),
        "schema_suffix": _pick("schema_suffix"),
        "tag_prefix": _pick("tag_prefix"),
        "tag_suffix": _pick("tag_suffix"),
        "table_id_type": _pick("table_id_type"),
        "boolean_format": _pick("boolean_format"),
        "date_format": _pick("date_format"),
        "timestamp_format": _pick("timestamp_format"),
        "classification_levels": _pick("classification_levels"),
        "housekeeping_columns": _pick("housekeeping_columns"),
        "history_tracking_columns": _pick("history_tracking_columns"),
    }


def _random_catalog_widgets(test_id=""):
    style = _pick("cataloging_style")
    tid = test_id.replace(" ", "_").lower()[:12] if test_id else ""
    prefix = f"t{tid}_" if tid else _pick("catalog_prefix")
    suffix = _pick("catalog_suffix")
    if style in ("Catalog per Domain", "Catalog per Division") and not prefix and not suffix:
        prefix = f"t{tid}_" if tid else "cat_"
    return {
        "cataloging_style": style,
        "catalog_prefix": prefix,
        "catalog_suffix": suffix,
    }


def _gen_session_id():
    return str((uuid.uuid4().int >> 64) & 9223372036854775807)


def _samples():
    return _pick("generate_samples_nonzero")


def _fmt_duration(seconds):
    s = int(seconds)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    if h > 0:
        return f"{h:02d}:{m:02d}:{sec:02d}"
    return f"{m:02d}:{sec:02d}"


def _ts():
    return datetime.now().strftime("%H:%M:%S")


def _banner(title, char="═", width=80):
    print(f"\n{char*width}")
    print(f"  [{_ts()}] {title}")
    print(f"{char*width}")


def _sub_banner(title, char="─", width=80):
    print(f"\n{char*width}")
    print(f"  [{_ts()}] {title}")
    print(f"{char*width}")


class TestResult:
    def __init__(self, test_name, test_label, status, duration_seconds, error_msg="", params=None):
        self.test_name = test_name
        self.test_label = test_label
        self.status = status
        self.duration_seconds = duration_seconds
        self.error_msg = error_msg
        self.params = params or {}


def _skip(test_name, test_label, reason):
    _banner(f"⏭️  SKIPPING: {test_label}", char="─")
    print(f"  Reason: {reason}")
    return TestResult(test_name, test_label, "SKIPPED", 0, reason)


# ═══════════════════════════════════════════════════════════════════
# MAIN — single entry point, ALL execution happens here
# ═══════════════════════════════════════════════════════════════════

def main():
    spark = SparkSession.builder.getOrCreate()

    # ── PHASE 1: READ WIDGETS ────────────────────────────────────
    _banner("PHASE 1: READING WIDGETS")
    w_business_name = dbutils.widgets.get("business_name").strip()
    w_business_description = dbutils.widgets.get("business_description").strip()
    w_model_vibes = dbutils.widgets.get("model_vibes").strip()
    w_catalog = dbutils.widgets.get("catalog").strip()

    print(f"  business_name:        {w_business_name}")
    print(f"  business_description: {w_business_description[:80]}{'...' if len(w_business_description) > 80 else ''}")
    print(f"  model_vibes:          {w_model_vibes[:80]}{'...' if len(w_model_vibes) > 80 else ''}")
    print(f"  catalog:              {w_catalog}")

    if not w_business_name:
        raise ValueError("Widget '01. Business Name' is required.")
    if not w_business_description:
        raise ValueError("Widget '02. Business Description' is required.")
    if not w_catalog:
        raise ValueError("Widget '04. Catalog' is required.")

    print(f"  [{_ts()}] All widgets validated OK")

    # ── PHASE 2: RESOLVE NOTEBOOK PATHS ────────────────────────
    _banner("PHASE 2: RESOLVING NOTEBOOK PATHS")
    tester_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    tester_dir = "/".join(tester_path.split("/")[:-1])
    print(f"  Tester path:    {tester_path}")
    print(f"  Tester dir:     {tester_dir}")

    agent_notebook = "./../agent/dbx_vibe_modelling_agent"
    runner_notebook = "./../runner/vibe_runner"
    print(f"  Agent notebook: {agent_notebook}")
    print(f"  Runner notebook: {runner_notebook}")

    sanitized_name = re.sub(r'[^a-z0-9_]', '_', w_business_name.lower().strip())
    sanitized_name = re.sub(r'_+', '_', sanitized_name).strip('_') or "unknown"
    print(f"  Sanitized name: {sanitized_name}")

    # ── PHASE 3: CLEAN SLATE — DROP & RECREATE CATALOG ───────────
    _banner("PHASE 3: CLEAN SLATE — DROP & RECREATE CATALOG")
    print(f"  [{_ts()}] Dropping catalog `{w_catalog}` (CASCADE)...")
    spark.sql(f"DROP CATALOG IF EXISTS `{w_catalog}` CASCADE")
    print(f"  [{_ts()}] Creating catalog `{w_catalog}`...")
    spark.sql(f"CREATE CATALOG `{w_catalog}`")
    print(f"  [{_ts()}] Creating schema `{w_catalog}`.`_metamodel`...")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{w_catalog}`.`_metamodel`")
    print(f"  [{_ts()}] Creating volume `{w_catalog}`.`_metamodel`.`vol_root`...")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{w_catalog}`.`_metamodel`.`vol_root`")
    print(f"  [{_ts()}] Catalog `{w_catalog}` ready")

    test_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    test_run_name = f"vibe_tester_{sanitized_name}_{test_run_timestamp}"
    volume_root = f"/Volumes/{w_catalog}/_metamodel/vol_root"
    log_output_dir = f"{volume_root}/logs/vibe_tester/{test_run_name}"

    try:
        dbutils.fs.mkdirs(log_output_dir)
    except Exception:
        pass
    print(f"  Log output dir: {log_output_dir}")

    def _read_file_content(path):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception:
            pass
        try:
            return dbutils.fs.head(path, 10 * 1024 * 1024)
        except Exception:
            return ""

    # ── Catalog-per-Domain/Division helpers ──────────────────────

    def _find_domain_catalogs(cat_prefix, cat_suffix):
        """Find all catalogs matching a prefix/suffix pattern (for Catalog per Domain/Division)."""
        try:
            all_cats = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
        except Exception:
            return []
        matched = []
        for c in all_cats:
            cl = c.lower()
            if cat_prefix and cl.startswith(cat_prefix.lower()):
                matched.append(c)
            elif cat_suffix and not cat_prefix and cl.endswith(cat_suffix.lower()):
                matched.append(c)
        return matched

    def _list_schemas_in_catalog(catalog):
        try:
            return [r[0] for r in spark.sql(f"SHOW SCHEMAS IN `{catalog}`").collect()]
        except Exception:
            return []

    def _count_tables_in_schema(catalog, schema):
        try:
            return len(spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`").collect())
        except Exception:
            return 0

    def _collect_schemas_and_tables(deploy_cat, cat_style, cat_prefix, cat_suffix):
        """Collect real schemas and table counts across all relevant catalogs."""
        all_real_schemas = []
        total_tables = 0
        catalogs_checked = []

        if cat_style in ("Catalog per Domain", "Catalog per Division"):
            domain_cats = _find_domain_catalogs(cat_prefix, cat_suffix)
            catalogs_checked = domain_cats
            for dc in domain_cats:
                schemas = _list_schemas_in_catalog(dc)
                real = [s for s in schemas if not s.startswith("_") and s not in ("default", "information_schema")]
                for s in real:
                    all_real_schemas.append(f"{dc}.{s}")
                    total_tables += _count_tables_in_schema(dc, s)
        else:
            catalogs_checked = [deploy_cat]
            schemas = _list_schemas_in_catalog(deploy_cat)
            real = [s for s in schemas if not s.startswith("_") and s not in ("default", "information_schema")]
            for s in real:
                all_real_schemas.append(f"{deploy_cat}.{s}")
                total_tables += _count_tables_in_schema(deploy_cat, s)

        return all_real_schemas, total_tables, catalogs_checked

    # ── Test builder closures ────────────────────────────────────

    def _build_base_model(test_name, test_label, scope_full, org_pool):
        convs = _random_conventions(test_name)
        cat_w = _random_catalog_widgets(test_name)
        return {
            "test_name": test_name,
            "test_label": test_label,
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "new base model",
                "model_version": "",
                "data_model_scopes": scope_full,
                "business_domains": "",
                "org_divisions": _pick(org_pool),
                "model_vibes": w_model_vibes,
                "deployment_catalog": w_catalog,
                "generate_samples": _samples(),
                "context_file": "",
                "vibe_session_id": _gen_session_id(),
                **convs,
                **cat_w,
            },
        }

    def _build_vibe_modeling(base_version="1"):
        vibes = w_model_vibes
        if not vibes:
            vibes = (
                "Add a new domain called 'Analytics' with products for dashboards and reports. "
                "Rename any domain that contains 'operations' to 'ops management'. "
                "Add a product called 'audit_log' under the first domain."
            )
        convs = _random_conventions("01_vibe")
        cat_w = _random_catalog_widgets("01_vibe")
        return {
            "test_name": "01_vibe_modeling_mvm",
            "test_label": f"Vibe Modeling (MVM v{base_version} → v{int(base_version)+1})",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "vibe modeling of version",
                "model_version": base_version,
                "data_model_scopes": "Minimum Viable Model - MVM",
                "business_domains": "",
                "org_divisions": _pick("org_divisions_mvm"),
                "model_vibes": vibes,
                "deployment_catalog": w_catalog,
                "generate_samples": _samples(),
                "context_file": "",
                "vibe_session_id": _gen_session_id(),
                **convs,
                **cat_w,
            },
        }

    def _build_resize(test_name, test_label, operation, source_version, scope_full, org_pool):
        convs = _random_conventions(test_name)
        cat_w = _random_catalog_widgets(test_name)
        return {
            "test_name": test_name,
            "test_label": test_label,
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": operation,
                "model_version": source_version,
                "data_model_scopes": scope_full,
                "business_domains": "",
                "org_divisions": _pick(org_pool),
                "model_vibes": w_model_vibes,
                "deployment_catalog": w_catalog,
                "generate_samples": _samples(),
                "context_file": "",
                "vibe_session_id": _gen_session_id(),
                **convs,
                **cat_w,
            },
        }

    def _find_model_json(version, scope):
        path = f"{volume_root}/business/{sanitized_name}/v{version}_{scope}/model.json"
        try:
            dbutils.fs.ls(path.rsplit("/", 1)[0])
            return path
        except Exception:
            pass
        base = f"{volume_root}/business/{sanitized_name}/v{version}_{scope}"
        try:
            items = dbutils.fs.ls(base)
            for item in items:
                if item.name == "model.json":
                    return item.path.replace("dbfs:", "")
        except Exception:
            pass
        try:
            docs_path = f"{base}/docs"
            items = dbutils.fs.ls(docs_path)
            for item in items:
                if item.name.endswith(".json") and "_data_model_v" in item.name:
                    return item.path.replace("dbfs:", "")
        except Exception:
            pass
        return path

    def _build_install(test_num, version, scope, deploy_suffix):
        scope_label = "ECM" if scope == "ecm" else "MVM"
        scope_full = "Expanded Coverage Model - ECM" if scope == "ecm" else "Minimum Viable Model - MVM"
        model_json = _find_model_json(version, scope)
        test_id = f"{test_num:02d}_inst{scope[0]}v{version}"
        convs = _random_conventions(test_id)
        cat_w = _random_catalog_widgets(test_id)
        deploy_cat = f"{w_catalog}_{sanitized_name}_{deploy_suffix}"
        return {
            "test_name": f"{test_num:02d}_install_{scope}_v{version}",
            "test_label": f"Install Model ({scope_label} v{version})",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "install model",
                "model_version": version,
                "data_model_scopes": scope_full,
                "business_domains": "",
                "org_divisions": "Operations and Business",
                "model_vibes": w_model_vibes,
                "deployment_catalog": deploy_cat,
                "generate_samples": _samples(),
                "context_file": model_json,
                "vibe_session_id": _gen_session_id(),
                **convs,
                **cat_w,
            },
            "_deploy_catalog": deploy_cat,
            "_model_json": model_json,
        }

    def _build_gen_samples(test_num, version, scope, deploy_catalog, model_json_path):
        scope_label = "ECM" if scope == "ecm" else "MVM"
        scope_full = "Expanded Coverage Model - ECM" if scope == "ecm" else "Minimum Viable Model - MVM"
        test_id = f"{test_num:02d}_smp{scope[0]}v{version}"
        convs = _random_conventions(test_id)
        cat_w = _random_catalog_widgets(test_id)
        return {
            "test_name": f"{test_num:02d}_generate_samples_{scope}_v{version}",
            "test_label": f"Generate Samples ({scope_label} v{version})",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "generate sample data",
                "model_version": version,
                "data_model_scopes": scope_full,
                "business_domains": "",
                "org_divisions": "Operations and Business",
                "model_vibes": w_model_vibes,
                "deployment_catalog": deploy_catalog,
                "generate_samples": _samples(),
                "context_file": model_json_path,
                "vibe_session_id": _gen_session_id(),
                **convs,
                **cat_w,
            },
        }

    def _build_uninstall(test_num, version, scope, deploy_catalog, test_id="", install_params=None):
        scope_label = "ECM" if scope == "ecm" else "MVM"
        scope_full = "Expanded Coverage Model - ECM" if scope == "ecm" else "Minimum Viable Model - MVM"
        tid = test_id.replace(" ", "_").lower()[:12] if test_id else ""
        cat_style = (install_params or {}).get("cataloging_style", "One Catalog")
        cat_prefix = (install_params or {}).get("catalog_prefix", "")
        cat_suffix = (install_params or {}).get("catalog_suffix", "")
        return {
            "test_name": f"{test_num:02d}_uninstall_{scope}_v{version}",
            "test_label": f"Uninstall Model ({scope_label} v{version})",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "uninstall model version",
                "model_version": version,
                "data_model_scopes": scope_full,
                "business_domains": "",
                "org_divisions": "Operations and Business",
                "model_vibes": w_model_vibes,
                "deployment_catalog": deploy_catalog,
                "generate_samples": "0",
                "context_file": "",
                "vibe_session_id": _gen_session_id(),
                "naming_convention": "snake_case",
                "primary_key_suffix": "_id",
                "schema_prefix": f"t{tid}_" if tid else "",
                "schema_suffix": "",
                "tag_prefix": "dbx_",
                "tag_suffix": "",
                "table_id_type": "BIGINT",
                "boolean_format": "Boolean (True/False)",
                "date_format": "yyyy-MM-dd",
                "timestamp_format": "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
                "classification_levels": "restricted=restricted, confidential=confidential, internal=Internal, public=public",
                "housekeeping_columns": "No",
                "history_tracking_columns": "No",
                "cataloging_style": cat_style,
                "catalog_prefix": cat_prefix,
                "catalog_suffix": cat_suffix,
            },
        }

    # ── Exit JSON parser ──────────────────────────────────────────

    def _parse_exit_json(raw_value):
        """Parse structured exit JSON from a notebook run. Returns (status_str, warnings_list).
        Handles None, non-JSON, non-dict JSON, null warnings, non-list warnings, non-string items."""
        if not raw_value:
            return "", []
        try:
            parsed = json.loads(raw_value)
        except (json.JSONDecodeError, ValueError, TypeError):
            return "", []
        if not isinstance(parsed, dict):
            return "", []
        raw_w = parsed.get("warnings")
        warnings = [str(w) for w in raw_w] if isinstance(raw_w, list) else []
        status = parsed.get("status", "")
        return status, warnings

    # ── Test runner with child log replay ────────────────────────

    def run_test(test_def, timeout_seconds=43200):
        test_name = test_def["test_name"]
        test_label = test_def["test_label"]
        params = test_def["params"]

        _banner(f"🧪 TEST START: {test_label}")
        print(f"  Operation:       {params.get('operation', 'N/A')}")
        print(f"  Model Scope:     {params.get('data_model_scopes', 'N/A')}")
        print(f"  Version:         {params.get('model_version', 'auto')}")
        print(f"  Naming:          {params.get('naming_convention', 'N/A')}")
        print(f"  PK Suffix:       {params.get('primary_key_suffix', 'N/A')}")
        print(f"  Schema Prefix:   {params.get('schema_prefix', '(none)')}")
        print(f"  Schema Suffix:   {params.get('schema_suffix', '(none)')}")
        print(f"  Tag Prefix:      {params.get('tag_prefix', '(none)')}")
        print(f"  Tag Suffix:      {params.get('tag_suffix', '(none)')}")
        print(f"  Cataloging:      {params.get('cataloging_style', 'N/A')}")
        print(f"  Catalog Prefix:  {params.get('catalog_prefix', '(none)')}")
        print(f"  Catalog Suffix:  {params.get('catalog_suffix', '(none)')}")
        print(f"  Samples:         {params.get('generate_samples', '0')}")
        print(f"  ID Type:         {params.get('table_id_type', 'N/A')}")
        print(f"  Boolean:         {params.get('boolean_format', 'N/A')}")
        print(f"  Date:            {params.get('date_format', 'N/A')}")
        print(f"  Timestamp:       {params.get('timestamp_format', 'N/A')}")
        print(f"  Housekeeping:    {params.get('housekeeping_columns', 'N/A')}")
        print(f"  History Track:   {params.get('history_tracking_columns', 'N/A')}")
        print(f"  Classification:  {params.get('classification_levels', 'N/A')[:60]}")
        print(f"  Org Divisions:   {params.get('org_divisions', 'N/A')}")
        print(f"  Deploy Catalog:  {params.get('deployment_catalog', 'N/A')}")
        print(f"  Context File:    {params.get('context_file', '(none)')}")
        vibes_val = params.get('model_vibes', '')
        print(f"  Model Vibes:     {vibes_val[:80]}{'...' if len(vibes_val) > 80 else ''}" if vibes_val else "  Model Vibes:     (none)")
        print(f"  Session ID:      {params.get('vibe_session_id', 'N/A')}")

        _sub_banner(f"⏳ RUNNING: {test_label} (calling {agent_notebook})")

        def _snapshot_error_logs():
            """Capture current error log sizes so we can detect NEW content after a test."""
            snapshot = {}
            log_base = f"{volume_root}/logs/{sanitized_name}"
            try:
                version_dirs = dbutils.fs.ls(log_base)
                for vdir in version_dirs:
                    if not vdir.isDir():
                        continue
                    try:
                        files = dbutils.fs.ls(vdir.path)
                        for f in files:
                            fname = f.name.lower()
                            if fname.endswith(".log") and ("_error_" in fname or fname == "error.log"):
                                fpath = f.path.replace("dbfs:", "")
                                snapshot[fpath] = f.size
                    except Exception:
                        pass
            except Exception:
                pass
            return snapshot

        def _extract_real_error(test_params, pre_snapshot):
            """Read ONLY new error log content written AFTER the test started."""
            version = test_params.get("model_version", "1") or "1"
            scope_raw = (test_params.get("data_model_scopes") or "").lower()
            scope = "ecm" if "ecm" in scope_raw else "mvm"
            op = (test_params.get("operation") or "").strip().lower()
            if op in ("shrink ecm",):
                scope = "mvm"
            elif op in ("enlarge mvm",):
                scope = "ecm"
            elif op == "vibe modeling of version":
                try:
                    version = str(int(float(version)) + 1)
                except (ValueError, TypeError):
                    pass
            log_base = f"{volume_root}/logs/{sanitized_name}"
            candidate_dirs = [f"v{version}_{scope}"]
            for candidate_ver in range(1, 20):
                tag = f"v{candidate_ver}_{scope}"
                if tag not in candidate_dirs:
                    candidate_dirs.append(tag)
            real_errors = []
            for vdir in candidate_dirs:
                error_log = f"{log_base}/{vdir}/{sanitized_name}_error_{vdir}.log"
                content = _read_file_content(error_log)
                if not content or not content.strip():
                    continue
                prev_size = pre_snapshot.get(error_log, 0)
                if len(content.encode("utf-8", errors="replace")) <= prev_size:
                    continue
                new_content = content[prev_size:] if prev_size > 0 else content
                if not new_content.strip():
                    continue
                for line in new_content.strip().splitlines()[-30:]:
                    ll = line.lower()
                    if any(kw in ll for kw in ("error", "critical", "failed", "exception", "traceback", "valueerror", "keyerror", "typeerror")):
                        real_errors.append(line.strip())
                if real_errors:
                    break
            return real_errors

        pre_snapshot = _snapshot_error_logs()
        start_time = time.time()
        try:
            result = dbutils.notebook.run(agent_notebook, timeout_seconds, params)
            duration = time.time() - start_time
            _exit_warnings = []
            _exit_status = ""
            if result:
                print(f"  Notebook exit value: {str(result)[:200]}")
                _exit_status, _exit_warnings = _parse_exit_json(result)
            _has_warnings = _exit_warnings or _exit_status == "success_with_warnings"
            if _has_warnings:
                _w_count = len(_exit_warnings) if _exit_warnings else 0
                _banner(f"⚠️  TEST PASSED WITH {_w_count} WARNING(S): {test_label} ({_fmt_duration(duration)})", char="─")
                for _ew in _exit_warnings:
                    print(f"  ⚠️  {_ew.strip()[:150]}")
                _w_msg = f"WARNINGS ({_w_count}): " + ("; ".join(w.strip()[:100] for w in _exit_warnings) if _exit_warnings else "status=success_with_warnings (no details)")
                return TestResult(test_name, test_label, "PASSED", duration,
                    error_msg=_w_msg,
                    params=params)
            else:
                _banner(f"✅ TEST PASSED: {test_label} ({_fmt_duration(duration)})", char="─")
            return TestResult(test_name, test_label, "PASSED", duration, params=params)
        except Exception as e:
            duration = time.time() - start_time
            raw_error = str(e)[:2000]
            real_errors = _extract_real_error(params, pre_snapshot)
            if real_errors:
                deduped = []
                seen = set()
                for line in real_errors[-10:]:
                    clean = line.split(" - ", 2)[-1].strip() if " - " in line else line.strip()
                    if clean not in seen:
                        seen.add(clean)
                        deduped.append(clean)
                actual_error_str = "\n    ".join(deduped[-5:])
                error_msg = f"ACTUAL ERROR (from child notebook log):\n    {actual_error_str}\n\nDATABRICKS WRAPPER:\n    {raw_error[:500]}"
            else:
                error_msg = raw_error
            _banner(f"❌ TEST FAILED: {test_label} ({_fmt_duration(duration)})", char="─")
            print(f"  Error: {error_msg[:1000]}")
            return TestResult(test_name, test_label, "FAILED", duration, error_msg, params=params)

    def _run_test_expect_failure(test_def, expected_keywords, success_msg, test_type="negative"):
        """Run a test that is EXPECTED to fail. If it fails with any expected keyword, mark PASSED."""
        result = run_test(test_def)
        full_err_upper = result.error_msg.upper() if result.error_msg else ""
        if result.status == "FAILED":
            matched = any(kw.upper() in full_err_upper for kw in expected_keywords)
            if matched:
                print(f"  ✅ {success_msg}")
                return TestResult(test_def["test_name"], test_def["test_label"], "PASSED", result.duration_seconds, success_msg, result.params)
            else:
                print(f"  ✅ {test_type} test: agent correctly rejected the request (failed as expected)")
                return TestResult(test_def["test_name"], test_def["test_label"], "PASSED", result.duration_seconds,
                    f"{test_type} test passed (agent failed as expected): {result.error_msg[:300]}", result.params)
        if result.status == "PASSED":
            _has_only_warnings = result.error_msg and result.error_msg.startswith("WARNINGS")
            if _has_only_warnings:
                return TestResult(test_def["test_name"], test_def["test_label"], "FAILED", result.duration_seconds,
                    f"Expected failure but notebook succeeded with warnings — {test_type} guard is broken! {result.error_msg[:200]}", result.params)
            return TestResult(test_def["test_name"], test_def["test_label"], "FAILED", result.duration_seconds,
                f"Expected failure but notebook succeeded — {test_type} guard is broken!", result.params)
        return result

    def run_test_expect_clash(test_def):
        return _run_test_expect_failure(
            test_def,
            expected_keywords=["PHYSICAL DEPLOYMENT CLASH", "CLASH DETECTED", "ALREADY EXIST", "CLASH"],
            success_msg="Correctly detected deployment clash (expected failure)",
            test_type="clash detection",
        )

    # ── Log merger ───────────────────────────────────────────────

    def merge_all_logs(test_results):
        _banner("PHASE 5: MERGING ALL LOGS")
        print(f"  Target: {log_output_dir}")

        log_base = f"{volume_root}/logs/{sanitized_name}"

        def _discover_log_files():
            """Scan volume for ALL log files, return {vdir_name: {info: path, error: path}}."""
            result = {}
            try:
                version_dirs = dbutils.fs.ls(log_base)
                for vdir in version_dirs:
                    if not vdir.isDir():
                        continue
                    vdir_name = vdir.name.rstrip("/")
                    entry = {"info": None, "error": None}
                    try:
                        files = dbutils.fs.ls(vdir.path)
                        for f in files:
                            fname = f.name.lower()
                            if fname.endswith(".log"):
                                fpath = f.path.replace("dbfs:", "")
                                if "_info_" in fname or fname == "info.log":
                                    entry["info"] = fpath
                                elif "_error_" in fname or fname == "error.log":
                                    entry["error"] = fpath
                    except Exception:
                        pass
                    if entry["info"] or entry["error"]:
                        result[vdir_name] = entry
            except Exception:
                pass
            return result

        def _test_to_vdirs(tr):
            """Map a TestResult to the volume log directory names it could have written to."""
            params = tr.params
            op = (params.get("operation") or "").strip().lower()
            version = params.get("model_version", "1") or "1"
            scope_raw = (params.get("data_model_scopes") or "").lower()
            scope = "ecm" if "ecm" in scope_raw else "mvm"
            candidates = []
            if op == "new base model":
                candidates.append(f"v1_{scope}")
                for v in range(2, 10):
                    candidates.append(f"v{v}_{scope}")
            elif op == "vibe modeling of version":
                try:
                    nxt = str(int(float(version)) + 1)
                except (ValueError, TypeError):
                    nxt = "2"
                candidates.append(f"v{nxt}_{scope}")
                for v in range(int(nxt) + 1, int(nxt) + 5):
                    candidates.append(f"v{v}_{scope}")
            elif op == "shrink ecm":
                tgt = "mvm"
                candidates.append(f"v{version}_{tgt}")
                for v in range(2, 10):
                    candidates.append(f"v{v}_{tgt}")
            elif op == "enlarge mvm":
                tgt = "ecm"
                candidates.append(f"v{version}_{tgt}")
                for v in range(2, 10):
                    candidates.append(f"v{v}_{tgt}")
            elif op in ("install model", "uninstall model version", "generate sample data"):
                candidates.append(f"v{version}_{scope}")
            return candidates

        _TRACE_PATTERNS = re.compile(
            r'^\s*(at |Traceback |File ["\']|Caused by:|'
            r'\.\.\. \d|py4j\.|com\.databricks\.|java\.|'
            r'at java\.|at py4j\.|at com\.|'
            r'at sun\.|at jdk\.)'
        )

        def _strip_traces(content):
            """Return content with stack trace lines removed."""
            if not content:
                return ""
            out = []
            skip_next_indent = False
            for line in content.splitlines():
                stripped = line.strip()
                if not stripped:
                    skip_next_indent = False
                    out.append(line)
                    continue
                if _TRACE_PATTERNS.match(stripped):
                    skip_next_indent = True
                    continue
                if skip_next_indent and line.startswith(("\t", "    ")) and not re.match(r'\s*\d{4}-\d{2}-\d{2}', line):
                    continue
                skip_next_indent = False
                out.append(line)
            return "\n".join(out)

        def _extract_error_summaries(content):
            """Extract ERROR/CRITICAL/WARNING lines as clean one-liners with icons."""
            if not content:
                return []
            lines = []
            for raw in content.splitlines():
                stripped = raw.strip()
                if not stripped:
                    continue
                if _TRACE_PATTERNS.match(stripped):
                    continue
                has_ts = bool(re.match(r'\d{4}-\d{2}-\d{2}', stripped))
                ll = stripped.lower()
                if has_ts and " - error - " in ll:
                    msg = stripped.split(" - ERROR - ", 1)[-1] if " - ERROR - " in stripped else stripped.split(" - error - ", 1)[-1]
                    lines.append(f"  ❌ ERROR: {msg.strip()}")
                elif has_ts and " - critical - " in ll:
                    msg = stripped.split(" - CRITICAL - ", 1)[-1] if " - CRITICAL - " in stripped else stripped.split(" - critical - ", 1)[-1]
                    lines.append(f"  🔴 CRITICAL: {msg.strip()}")
                elif has_ts and " - warning - " in ll:
                    msg = stripped.split(" - WARNING - ", 1)[-1] if " - WARNING - " in stripped else stripped.split(" - warning - ", 1)[-1]
                    lines.append(f"  ⚠️  WARNING: {msg.strip()}")
            return lines

        vdir_map = _discover_log_files()
        claimed_vdirs = set()

        total_info_files = sum(1 for v in vdir_map.values() if v["info"])
        total_error_files = sum(1 for v in vdir_map.values() if v["error"])
        print(f"  Discovered {len(vdir_map)} log directories on volume")
        print(f"  Found {total_info_files} info log file(s), {total_error_files} error log file(s)")
        for vd, paths in sorted(vdir_map.items()):
            print(f"    {vd}/  info={'✓' if paths['info'] else '✗'}  error={'✓' if paths['error'] else '✗'}")

        merged_info = []
        merged_error = []

        def _append_header(lines, label):
            lines.append(f"{'='*80}")
            lines.append(f"VIBE TESTER - MERGED {label} LOGS")
            lines.append(f"Run: {test_run_name}")
            lines.append(f"Business: {w_business_name}")
            lines.append(f"Catalog: {w_catalog}")
            lines.append(f"Agent Notebook: {agent_notebook}")
            lines.append(f"Runner Notebook: {runner_notebook}")
            lines.append(f"Timestamp: {datetime.now().isoformat()}")
            lines.append(f"Total Tests: {len(test_results)}")
            lines.append(f"{'='*80}\n")

        _append_header(merged_info, "INFO")
        _append_header(merged_error, "ERROR")

        for tr in test_results:
            status_icon = "✅" if tr.status == "PASSED" else ("❌" if tr.status == "FAILED" else "⏭️")
            dur_str = _fmt_duration(tr.duration_seconds)
            header = f"\n{'─'*60}\n{status_icon} [TEST: {tr.test_name}] {tr.test_label} — {tr.status} ({dur_str})\n{'─'*60}"

            merged_info.append(header)
            merged_error.append(header)

            if tr.status == "SKIPPED":
                merged_info.append(f"  ⏭️  Skipped: {tr.error_msg}\n")
                merged_error.append(f"  ⏭️  Skipped: {tr.error_msg}\n")
                continue

            candidates = _test_to_vdirs(tr)
            matched_vdirs = [c for c in candidates if c in vdir_map]

            info_content_parts = []
            error_content_parts = []

            for vd in matched_vdirs:
                claimed_vdirs.add(vd)
                paths = vdir_map[vd]
                if paths["info"]:
                    c = _read_file_content(paths["info"])
                    if c and c.strip():
                        info_content_parts.append((vd, c))
                if paths["error"]:
                    c = _read_file_content(paths["error"])
                    if c and c.strip():
                        error_content_parts.append((vd, c))

            if info_content_parts:
                for vd, content in info_content_parts:
                    short = f"logs/{sanitized_name}/{vd}/"
                    merged_info.append(f"\n  📄 [SOURCE: {short}]")
                    merged_info.append(_strip_traces(content))
            else:
                if tr.status != "PASSED":
                    merged_info.append(f"  (no info log found on volume for this test)")

            if error_content_parts:
                for vd, content in error_content_parts:
                    short = f"logs/{sanitized_name}/{vd}/"
                    clean = _strip_traces(content)
                    merged_error.append(f"\n  📄 [SOURCE: {short}]")
                    merged_error.append(clean)
                    error_summaries = _extract_error_summaries(content)
                    if error_summaries:
                        merged_info.append(f"\n  ⬇️  ERRORS/WARNINGS from {short}:")
                        merged_info.extend(error_summaries)

            if tr.status == "FAILED":
                raw_err = tr.error_msg or ""
                clean_lines = []
                for line in raw_err.splitlines():
                    s = line.strip()
                    if not s:
                        continue
                    if _TRACE_PATTERNS.match(s):
                        continue
                    clean_lines.append(s)
                if clean_lines:
                    msg_text = "\n    ".join(clean_lines[:10])
                    merged_error.append(f"\n  ❌ TEST FAILURE MESSAGE:\n    {msg_text}")
                    if not error_content_parts:
                        merged_info.append(f"\n  ❌ TEST FAILURE:\n    {msg_text}")

            merged_info.append("")
            merged_error.append("")

        unclaimed = set(vdir_map.keys()) - claimed_vdirs
        if unclaimed:
            unclaimed_header = f"\n{'─'*60}\nUNCLAIMED LOG FILES (not matched to any test)\n{'─'*60}"
            merged_info.append(unclaimed_header)
            merged_error.append(unclaimed_header)
            for vd in sorted(unclaimed):
                paths = vdir_map[vd]
                if paths["info"]:
                    c = _read_file_content(paths["info"])
                    if c and c.strip():
                        short = f"logs/{sanitized_name}/{vd}/"
                        merged_info.append(f"\n  📄 [UNCLAIMED SOURCE: {short}]")
                        merged_info.append(_strip_traces(c))
                if paths["error"]:
                    c = _read_file_content(paths["error"])
                    if c and c.strip():
                        short = f"logs/{sanitized_name}/{vd}/"
                        merged_error.append(f"\n  📄 [UNCLAIMED SOURCE: {short}]")
                        merged_error.append(_strip_traces(c))
                        error_summaries = _extract_error_summaries(c)
                        if error_summaries:
                            merged_info.append(f"\n  ⬇️  ERRORS/WARNINGS from {short}:")
                            merged_info.extend(error_summaries)

        _sub_banner("Writing merged log files")
        for path, content_lines in [
            (f"{log_output_dir}/merged_info.log", merged_info),
            (f"{log_output_dir}/merged_error.log", merged_error),
        ]:
            try:
                dbutils.fs.put(path, "\n".join(content_lines), overwrite=True)
                print(f"  ✅ Written: {path}")
            except Exception as e:
                print(f"  ❌ Failed: {path} — {e}")

        passed = sum(1 for t in test_results if t.status == "PASSED")
        failed = sum(1 for t in test_results if t.status == "FAILED")
        skipped = sum(1 for t in test_results if t.status == "SKIPPED")
        warned = sum(1 for t in test_results if t.status == "PASSED" and t.error_msg and t.error_msg.startswith("WARNINGS"))

        summary = []
        summary.append(f"{'='*80}")
        summary.append(f"VIBE TESTER SUMMARY — {test_run_name}")
        summary.append(f"{'='*80}")
        summary.append(f"Business:    {w_business_name}")
        summary.append(f"Catalog:     {w_catalog}")
        summary.append(f"Agent NB:    {agent_notebook}")
        summary.append(f"Runner NB:   {runner_notebook}")
        summary.append(f"Started:     {test_run_timestamp}")
        summary.append(f"Finished:    {datetime.now().strftime('%Y%m%d_%H%M%S')}")
        summary.append(f"{'='*80}\n")
        _warned_str = f"  |  WARNED: {warned}" if warned else ""
        summary.append(f"TOTAL: {len(test_results)}  |  PASSED: {passed}  |  FAILED: {failed}  |  SKIPPED: {skipped}{_warned_str}")
        summary.append(f"{'─'*80}\n")

        for tr in test_results:
            icon = "✅" if tr.status == "PASSED" else ("❌" if tr.status == "FAILED" else "⏭️")
            summary.append(f"  {icon} [{tr.test_name}] {tr.test_label}")
            summary.append(f"     Status:   {tr.status}")
            summary.append(f"     Duration: {_fmt_duration(tr.duration_seconds)}")
            key_params = {k: v for k, v in tr.params.items() if k in (
                "operation", "data_model_scopes", "model_version", "naming_convention",
                "table_id_type", "boolean_format", "cataloging_style", "generate_samples",
                "primary_key_suffix", "schema_prefix", "schema_suffix", "tag_prefix",
                "housekeeping_columns", "history_tracking_columns", "deployment_catalog",
                "org_divisions", "catalog_prefix", "catalog_suffix",
            ) and v}
            summary.append(f"     Params:   {json.dumps(key_params)}")
            if tr.error_msg:
                clean_err = tr.error_msg
                for line in tr.error_msg.splitlines():
                    if _TRACE_PATTERNS.match(line.strip()):
                        clean_err = clean_err.replace(line, "")
                clean_err = re.sub(r'\n{3,}', '\n\n', clean_err).strip()
                _err_label = "Warnings:" if tr.status == "PASSED" and tr.error_msg.startswith("WARNINGS") else "Error:   "
                summary.append(f"     {_err_label} {clean_err[:500]}")
            summary.append("")

        summary.extend([f"\n{'='*80}", f"Log files at: {log_output_dir}", f"{'='*80}"])
        try:
            dbutils.fs.put(f"{log_output_dir}/test_summary.log", "\n".join(summary), overwrite=True)
            print(f"  ✅ Written: {log_output_dir}/test_summary.log")
        except Exception as e:
            print(f"  ❌ Failed: {log_output_dir}/test_summary.log — {e}")

        return passed, failed, skipped, warned

    # ═════════════════════════════════════════════════════════════
    # PHASE 4: RUN ALL TESTS
    # ═════════════════════════════════════════════════════════════

    print(f"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║  🧪 VIBE MODELLING AGENT — COMPREHENSIVE TESTER                                  ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  Business:     {w_business_name:<63} ║
║  Catalog:      {w_catalog:<63} ║
║  Agent NB:     {agent_notebook[:63]:<63} ║
║  Runner NB:    {runner_notebook[:63]:<63} ║
║  Test Run:     {test_run_name:<63} ║
║  Log Output:   {log_output_dir[:63]:<63} ║
║                                                                                  ║
║  STRATEGY: Runner creates ECM+MVM → ALL ops + edge cases + negative tests        ║
║  ARTIFACTS: generate_samples always > 0 — full artifact generation               ║
║  TESTS: 23 total (1 runner + 11 core + 5 verification + 3 edge + 2 neg + 1 re)  ║
║  LOGS: consolidated into merged_info.log + merged_error.log on volume            ║
╚══════════════════════════════════════════════════════════════════════════════════╝
""")

    _banner("PHASE 4: RUNNING ALL TESTS (23 tests)")
    overall_start = time.time()
    R = []

    # ─────────────────────────────────────────────────────────────
    # TEST 00: VIBE RUNNER — creates ECM v1 and MVM v1 via the
    #          full runner pipeline (temp JSON → job → install)
    # ─────────────────────────────────────────────────────────────
    _banner("TEST 00: Vibe Runner (create ECM + MVM via runner pipeline)", char="▓")
    start_00 = time.time()
    ecm_ok = False
    mvm_ok = False
    try:
        _sub_banner("Creating temp industry JSON for vibe runner")
        temp_json_data = {
            "widget_values": {
                "business_domains": "",
                "org_divisions": "Operations, Business and Corporate",
                "model_vibes": "",
                "cataloging_style": "One Catalog",
                "catalog_prefix": "",
                "catalog_suffix": "",
                "generate_samples": "0",
                "naming_convention": "snake_case",
                "primary_key_suffix": "_id",
                "schema_prefix": "",
                "schema_suffix": "",
                "tag_prefix": "dbx_",
                "tag_suffix": "",
                "table_id_type": "BIGINT",
                "boolean_format": "Boolean (True/False)",
                "date_format": "yyyy-MM-dd",
                "timestamp_format": "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
                "classification_levels": "restricted=restricted, confidential=confidential, internal=Internal, public=public",
                "housekeeping_columns": "No",
                "history_tracking_columns": "No",
            },
            "businesses": [
                {
                    "name": w_business_name,
                    "description": w_business_description,
                    "model_vibes": w_model_vibes if w_model_vibes else "maximum of 2 domains, and 8 tables for any model you generate",
                }
            ],
        }
        temp_json_str = json.dumps(temp_json_data, indent=2)
        temp_json_volume_path = f"{volume_root}/temp_test_industries.json"
        temp_json_fuse_path = f"/Volumes/{w_catalog}/_metamodel/vol_root/temp_test_industries.json"
        dbutils.fs.put(temp_json_volume_path, temp_json_str, overwrite=True)
        print(f"  Temp JSON written to: {temp_json_volume_path}")
        print(f"  FUSE path for runner: {temp_json_fuse_path}")
        print(f"  Business:  {w_business_name}")

        _sub_banner(f"⏳ Calling vibe runner: {runner_notebook}")
        runner_result = dbutils.notebook.run(runner_notebook, 43200, {
            "business_context": temp_json_fuse_path,
            "dry_run": "no",
            "ping_interval": "30s",
        })
        dur_00 = time.time() - start_00
        print(f"  Runner exit value: {str(runner_result)[:200]}")

        _sub_banner("Verifying runner output catalogs")
        runner_sanitized = _sanitize_name_for_runner(w_business_name)
        runner_ecm_catalog = f"{runner_sanitized}_ecm_v1"
        runner_mvm_catalog = f"{runner_sanitized}_mvm_v1"
        print(f"  Expected ECM catalog: {runner_ecm_catalog}")
        print(f"  Expected MVM catalog: {runner_mvm_catalog}")

        all_cats = [r[0].lower() for r in spark.sql("SHOW CATALOGS").collect()]
        ecm_exists = runner_ecm_catalog.lower() in all_cats
        mvm_exists = runner_mvm_catalog.lower() in all_cats
        print(f"  ECM catalog exists: {ecm_exists}")
        print(f"  MVM catalog exists: {mvm_exists}")

        if not ecm_exists or not mvm_exists:
            missing = []
            if not ecm_exists:
                missing.append(runner_ecm_catalog)
            if not mvm_exists:
                missing.append(runner_mvm_catalog)
            raise ValueError(f"Runner completed but catalogs missing: {', '.join(missing)}")

        _sub_banner("Verifying runner catalogs have metamodel data (not just empty shells)")
        for label, cat in [("ECM", runner_ecm_catalog), ("MVM", runner_mvm_catalog)]:
            try:
                schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN `{cat}`").collect()]
                if "_metamodel" not in schemas:
                    raise ValueError(f"{label} catalog '{cat}' has no _metamodel schema — runner pipeline likely failed")
                biz_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM `{cat}`.`_metamodel`.`business`").first().c
                if biz_cnt == 0:
                    raise ValueError(f"{label} catalog '{cat}' _metamodel.business is empty — runner pipeline likely failed")
                print(f"  {label} catalog verified: _metamodel.business has {biz_cnt} row(s)")
            except ValueError:
                raise
            except Exception as e:
                raise ValueError(f"{label} catalog '{cat}' verification failed: {str(e)[:300]}")

        _sub_banner("Copying metamodel data from runner catalogs to test catalog")
        mm_tables = ["business", "domain", "product", "attribute"]
        src_mm = f"`{runner_ecm_catalog}`.`_metamodel`"
        dst_mm = f"`{w_catalog}`.`_metamodel`"

        for tbl in mm_tables:
            try:
                existing = spark.sql(f"SHOW TABLES IN {dst_mm}").collect()
                tbl_exists = any(r.tableName == tbl for r in existing)
            except Exception:
                tbl_exists = False
            try:
                if not tbl_exists:
                    spark.sql(f"CREATE TABLE {dst_mm}.{tbl} AS SELECT * FROM {src_mm}.{tbl}")
                else:
                    spark.sql(f"INSERT INTO {dst_mm}.{tbl} SELECT * FROM {src_mm}.{tbl}")
                cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {dst_mm}.{tbl}").first().c
                print(f"  {tbl}: copied ECM rows → {cnt} total in {dst_mm}")
            except Exception as e:
                print(f"  {tbl}: ECM copy failed — {str(e)[:200]}")

        mvm_mm = f"`{runner_mvm_catalog}`.`_metamodel`"
        for tbl in mm_tables:
            try:
                spark.sql(f"INSERT INTO {dst_mm}.{tbl} SELECT * FROM {mvm_mm}.{tbl}")
                cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {dst_mm}.{tbl}").first().c
                print(f"  {tbl}: copied MVM rows → {cnt} total in {dst_mm}")
            except Exception as e:
                print(f"  {tbl}: MVM copy failed — {str(e)[:200]}")

        _sub_banner("Fixing metamodel tables for test catalog context")
        try:
            spark.sql(f"""
                UPDATE {dst_mm}.business
                SET version  = regexp_replace(version, '_(ecm|mvm)$', ''),
                    catalog  = '{w_catalog}',
                    location = regexp_replace(location, '^/Volumes/[^/]+/', '/Volumes/{w_catalog}/')
            """)
            fixed_rows = spark.sql(f"SELECT version, model_scope, catalog FROM {dst_mm}.business").collect()
            for row in fixed_rows:
                print(f"  business: version={row.version}, scope={row.model_scope}, catalog={row.catalog}")
        except Exception as e:
            print(f"  ⚠️ Business table fix failed — {str(e)[:300]}")
        for mm_tbl in ["domain", "product", "attribute"]:
            try:
                cols = [c.name for c in spark.sql(f"DESCRIBE {dst_mm}.{mm_tbl}").collect()]
                updates = ["version = regexp_replace(version, '_(ecm|mvm)$', '')"]
                if "catalog" in cols:
                    updates.append(f"catalog = '{w_catalog}'")
                spark.sql(f"UPDATE {dst_mm}.{mm_tbl} SET {', '.join(updates)}")
                cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {dst_mm}.{mm_tbl}").first().c
                print(f"  {mm_tbl}: version fixed ({cnt} rows)")
            except Exception as e:
                print(f"  ⚠️ {mm_tbl} fix failed — {str(e)[:200]}")

        _sub_banner("Copying volume files from runner catalogs to test catalog")
        ecm_vol_src = f"/Volumes/{runner_ecm_catalog}/_metamodel/vol_root/business/{runner_sanitized}/"
        mvm_vol_src = f"/Volumes/{runner_mvm_catalog}/_metamodel/vol_root/business/{runner_sanitized}/"
        vol_dst = f"{volume_root}/business/{sanitized_name}/"
        for label, vol_src in [("ECM", ecm_vol_src), ("MVM", mvm_vol_src)]:
            try:
                dbutils.fs.cp(vol_src, vol_dst, recurse=True)
                print(f"  {label} volume files copied: {vol_src} → {vol_dst}")
            except Exception as e:
                print(f"  {label} volume copy failed — {str(e)[:200]}")

        _sub_banner("Validating and patching model.json files on volume")
        ecm_model_path = f"{volume_root}/business/{sanitized_name}/v1_ecm/model.json"
        mvm_model_path = f"{volume_root}/business/{sanitized_name}/v1_mvm/model.json"
        for label, mpath in [("ECM", ecm_model_path), ("MVM", mvm_model_path)]:
            fuse_path = mpath.replace("dbfs:", "")
            try:
                dbutils.fs.ls(mpath.rsplit("/", 1)[0])
                print(f"  {label} model.json directory verified: {mpath.rsplit('/', 1)[0]}")
            except Exception:
                print(f"  ⚠️  {label} model.json directory NOT found at {mpath.rsplit('/', 1)[0]} — downstream tests may fail")
                continue
            try:
                with open(fuse_path, "r") as mf:
                    model_data = json.loads(mf.read())
                old_ver = model_data.get("version", "")
                if re.search(r'_(ecm|mvm)$', str(old_ver)):
                    model_data["version"] = re.sub(r'_(ecm|mvm)$', '', str(old_ver))
                    print(f"  {label} model.json version patched: {old_ver} → {model_data['version']}")
                if model_data.get("catalog") and model_data["catalog"] != w_catalog:
                    model_data["catalog"] = w_catalog
                if model_data.get("location"):
                    model_data["location"] = re.sub(r'^/Volumes/[^/]+/', f'/Volumes/{w_catalog}/', model_data["location"])
                with open(fuse_path, "w") as mf:
                    mf.write(json.dumps(model_data, indent=2))
                print(f"  {label} model.json patched and written back")
            except Exception as e:
                print(f"  ⚠️  {label} model.json patch failed — {str(e)[:200]}")

        try:
            dbutils.fs.rm(temp_json_volume_path)
        except Exception:
            pass

        ecm_ok = True
        mvm_ok = True
        _banner(f"✅ TEST PASSED: Vibe Runner (ECM + MVM) ({_fmt_duration(dur_00)})", char="─")
        R.append(TestResult("00_vibe_runner", "Vibe Runner (create ECM + MVM via runner pipeline)", "PASSED", dur_00, params={}))

    except Exception as e:
        dur_00 = time.time() - start_00
        error_msg = str(e)[:2000]
        _banner(f"❌ TEST FAILED: Vibe Runner ({_fmt_duration(dur_00)})", char="─")
        print(f"  Error: {error_msg[:1000]}")
        R.append(TestResult("00_vibe_runner", "Vibe Runner (create ECM + MVM via runner pipeline)", "FAILED", dur_00, error_msg, params={}))

    # 01: Vibe Modeling (MVM v1 → v2)
    _banner("TEST 01: Vibe Modeling (MVM v1 → v2)", char="▓")
    if mvm_ok:
        R.append(run_test(_build_vibe_modeling(base_version="1")))
    else:
        R.append(_skip("01_vibe_modeling_mvm", "Vibe Modeling (MVM v1 → v2)", "vibe runner (test 00) failed — no MVM base model"))
    vibe_ok = R[-1].status == "PASSED"

    # 02: Shrink ECM v1 → MVM
    _banner("TEST 02: Shrink ECM v1 → MVM", char="▓")
    if ecm_ok:
        R.append(run_test(_build_resize("02_shrink_ecm_to_mvm", "Shrink ECM v1 → MVM", "shrink ecm", "1", "Expanded Coverage Model - ECM", "org_divisions_mvm")))
    else:
        R.append(_skip("02_shrink_ecm_to_mvm", "Shrink ECM v1 → MVM", "vibe runner (test 00) failed — no ECM base model"))

    # 03: Enlarge MVM v1 → ECM
    _banner("TEST 03: Enlarge MVM v1 → ECM", char="▓")
    if mvm_ok:
        R.append(run_test(_build_resize("03_enlarge_mvm_to_ecm", "Enlarge MVM v1 → ECM", "enlarge mvm", "1", "Minimum Viable Model - MVM", "org_divisions_ecm")))
    else:
        R.append(_skip("03_enlarge_mvm_to_ecm", "Enlarge MVM v1 → ECM", "vibe runner (test 00) failed — no MVM base model"))

    # 04: Install MVM v1
    _banner("TEST 04: Install Model (MVM v1)", char="▓")
    mvm1_deploy_cat = ""
    mvm1_model_json = ""
    mvm1_install_params = {}
    if mvm_ok:
        td = _build_install(4, "1", "mvm", "mvm_v1_installed")
        mvm1_deploy_cat = td["_deploy_catalog"]
        mvm1_model_json = td["_model_json"]
        mvm1_install_params = dict(td["params"])
        R.append(run_test(td))
    else:
        R.append(_skip("04_install_mvm_v1", "Install Model (MVM v1)", "vibe runner (test 00) failed — no MVM base model"))
    mvm1_installed = R[-1].status == "PASSED"

    # 04b: Clash Detection — reinstall MVM v1 with SAME conventions as test 06
    _banner("TEST 04b: Clash Detection (reinstall MVM v1)", char="▓")
    if mvm1_installed and mvm1_model_json:
        td_clash = {
            "test_name": "04b_clash_detection_mvm_v1",
            "test_label": "Clash Detection (reinstall MVM v1)",
            "params": dict(mvm1_install_params),
        }
        td_clash["params"]["deployment_catalog"] = mvm1_deploy_cat
        td_clash["params"]["context_file"] = mvm1_model_json
        td_clash["params"]["vibe_session_id"] = _gen_session_id()
        R.append(run_test_expect_clash(td_clash))
    else:
        R.append(_skip("04b_clash_detection_mvm_v1", "Clash Detection (reinstall MVM v1)", "install MVM v1 (test 04) failed"))

    # 04c: Verify Install Physical — schemas/tables/cataloging_style structure
    _banner("TEST 04c: Verify Install Physical (MVM v1)", char="▓")
    if mvm1_installed:
        start_04c = time.time()
        issues_04c = []
        deploy_04c = mvm1_deploy_cat
        cat_style_04c = mvm1_install_params.get("cataloging_style", "One Catalog")
        cat_prefix_04c = mvm1_install_params.get("catalog_prefix", "")
        cat_suffix_04c = mvm1_install_params.get("catalog_suffix", "")

        all_schemas_04c, total_tables_04c, cats_checked_04c = _collect_schemas_and_tables(
            deploy_04c, cat_style_04c, cat_prefix_04c, cat_suffix_04c
        )

        if not all_schemas_04c:
            issues_04c.append(f"No user schemas found (style={cat_style_04c}, prefix='{cat_prefix_04c}', suffix='{cat_suffix_04c}', catalogs checked: {cats_checked_04c})")
        if total_tables_04c == 0 and all_schemas_04c:
            issues_04c.append(f"Found {len(all_schemas_04c)} schemas but 0 tables — physical DDL may have failed silently")

        if cat_style_04c in ("Catalog per Domain", "Catalog per Division"):
            domain_cats = _find_domain_catalogs(cat_prefix_04c, cat_suffix_04c)
            print(f"  {cat_style_04c} check: {len(domain_cats)} catalogs found (prefix='{cat_prefix_04c}', suffix='{cat_suffix_04c}')")
            if not domain_cats:
                issues_04c.append(f"cataloging_style='{cat_style_04c}' but found 0 catalogs with prefix='{cat_prefix_04c}' suffix='{cat_suffix_04c}'")
        elif cat_style_04c == "One Catalog":
            print(f"  One-Catalog check: {len(all_schemas_04c)} schemas in {deploy_04c}")

        dur_04c = time.time() - start_04c
        print(f"  Deploy catalog:     {deploy_04c}")
        print(f"  Cataloging style:   {cat_style_04c}")
        print(f"  Schemas found:      {len(all_schemas_04c)} {all_schemas_04c[:10]}")
        print(f"  Tables found:       {total_tables_04c}")
        if issues_04c:
            for i in issues_04c:
                print(f"  ISSUE: {i}")
            R.append(TestResult("04c_verify_install", "Verify Install Physical (MVM v1)", "FAILED", dur_04c, "\n".join(issues_04c), mvm1_install_params))
        else:
            R.append(TestResult("04c_verify_install", "Verify Install Physical (MVM v1)", "PASSED", dur_04c, params=mvm1_install_params))
    else:
        R.append(_skip("04c_verify_install", "Verify Install Physical (MVM v1)", "install MVM v1 (test 04) failed"))

    # 05: Generate Samples on installed MVM v1
    _banner("TEST 05: Generate Samples (MVM v1)", char="▓")
    if mvm1_installed:
        R.append(run_test(_build_gen_samples(5, "1", "mvm", mvm1_deploy_cat, mvm1_model_json)))
    else:
        R.append(_skip("05_generate_samples_mvm_v1", "Generate Samples (MVM v1)", "install MVM v1 (test 04) failed"))
    mvm1_samples_ok = R[-1].status == "PASSED"

    # 05b: Verify Sample Data — tables actually have rows
    _banner("TEST 05b: Verify Sample Data (MVM v1)", char="▓")
    if mvm1_samples_ok:
        start_05b = time.time()
        deploy_05b = mvm1_deploy_cat
        cat_style_05b = mvm1_install_params.get("cataloging_style", "One Catalog")
        cat_prefix_05b = mvm1_install_params.get("catalog_prefix", "")
        cat_suffix_05b = mvm1_install_params.get("catalog_suffix", "")

        all_schemas_05b, _, cats_05b = _collect_schemas_and_tables(
            deploy_05b, cat_style_05b, cat_prefix_05b, cat_suffix_05b
        )

        populated_05b = 0
        empty_tables_05b = []
        total_tables_05b = 0
        for fq_schema in all_schemas_05b[:40]:
            parts = fq_schema.split(".", 1)
            cat_05b, schema_05b = parts[0], parts[1]
            try:
                tables = spark.sql(f"SHOW TABLES IN `{cat_07b}`.`{schema_07b}`").collect()
            except Exception:
                continue
            for trow in tables:
                tname = trow.tableName
                total_tables_05b += 1
                try:
                    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM `{cat_07b}`.`{schema_07b}`.`{tname}`").first().c
                    if cnt > 0:
                        populated_05b += 1
                    else:
                        empty_tables_05b.append(f"{cat_07b}.{schema_07b}.{tname}")
                except Exception:
                    empty_tables_05b.append(f"{cat_07b}.{schema_07b}.{tname} (read error)")

        dur_05b = time.time() - start_05b
        pop_rate_05b = (populated_05b / total_tables_05b * 100) if total_tables_05b else 0
        print(f"  Tables checked:    {total_tables_07b}")
        print(f"  Tables with data:  {populated_07b} ({pop_rate_07b:.1f}%)")
        if empty_tables_05b[:10]:
            print(f"  Empty tables:      {empty_tables_05b[:10]}")
        if pop_rate_05b >= 50:
            R.append(TestResult("05b_verify_samples", "Verify Sample Data (MVM v1)", "PASSED", dur_05b, params={}))
        else:
            R.append(TestResult("05b_verify_samples", "Verify Sample Data (MVM v1)", "FAILED", dur_05b,
                f"Only {populated_07b}/{total_tables_07b} tables ({pop_rate_07b:.1f}%) have sample data. Empty: {empty_tables_05b[:20]}", {}))
    else:
        R.append(_skip("05b_verify_samples", "Verify Sample Data (MVM v1)", "generate samples v1 (test 05) failed"))

    # 06: Uninstall MVM v1
    _banner("TEST 06: Uninstall Model (MVM v1)", char="▓")
    if mvm1_installed:
        R.append(run_test(_build_uninstall(6, "1", "mvm", mvm1_deploy_cat, test_id="06_uninst1", install_params=mvm1_install_params)))
    else:
        R.append(_skip("06_uninstall_mvm_v1", "Uninstall Model (MVM v1)", "install MVM v1 (test 04) failed"))
    mvm1_uninstall_ok = R[-1].status == "PASSED"

    # 06b: Verify Uninstall Cleanup — no leftover tables
    _banner("TEST 06b: Verify Uninstall Cleanup (MVM v1)", char="▓")
    if mvm1_uninstall_ok:
        start_06b = time.time()
        deploy_06b = mvm1_deploy_cat
        cat_exists_06b = True
        try:
            schemas_06b = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN `{deploy_08b}`").collect()]
        except Exception:
            try:
                schemas_06b = [r.namespace for r in spark.sql(f"SHOW SCHEMAS IN `{deploy_08b}`").collect()]
            except Exception:
                schemas_06b = []
                cat_exists_06b = False
        real_schemas_06b = [s for s in schemas_06b if not s.startswith("_") and s not in ("default", "information_schema")]

        leftover_tables_06b = 0
        for schema in real_schemas_06b[:20]:
            try:
                leftover_tables_06b += len(spark.sql(f"SHOW TABLES IN `{deploy_08b}`.`{schema}`").collect())
            except Exception:
                pass

        dur_06b = time.time() - start_06b
        print(f"  Deploy catalog:     {deploy_08b}")
        print(f"  Catalog still exists: {cat_exists_08b}")
        print(f"  Remaining schemas:  {len(real_schemas_06b)} {real_schemas_06b[:10]}")
        print(f"  Remaining tables:   {leftover_tables_08b}")
        if leftover_tables_06b == 0:
            R.append(TestResult("06b_verify_uninstall", "Verify Uninstall Cleanup (MVM v1)", "PASSED", dur_06b, params={}))
        else:
            R.append(TestResult("06b_verify_uninstall", "Verify Uninstall Cleanup (MVM v1)", "FAILED", dur_06b,
                f"Uninstall left behind {leftover_tables_08b} tables across {len(real_schemas_06b)} schemas in {deploy_08b}", {}))
    else:
        R.append(_skip("06b_verify_uninstall", "Verify Uninstall Cleanup (MVM v1)", "uninstall MVM v1 (test 06) failed"))

    # 06c: Reinstall After Uninstall — full cycle test
    _banner("TEST 06c: Reinstall After Uninstall (MVM v1)", char="▓")
    if mvm1_uninstall_ok and mvm1_install_params and mvm1_model_json:
        reinstall_params = dict(mvm1_install_params)
        reinstall_params["deployment_catalog"] = mvm1_deploy_cat
        reinstall_params["context_file"] = mvm1_model_json
        reinstall_params["vibe_session_id"] = _gen_session_id()
        td_reinstall = {
            "test_name": "06c_reinstall",
            "test_label": "Reinstall After Uninstall (MVM v1)",
            "params": reinstall_params,
        }
        R.append(run_test(td_reinstall))
        if R[-1].status == "PASSED":
            try:
                re_schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN `{mvm1_deploy_cat}`").collect()]
            except Exception:
                try:
                    re_schemas = [r.namespace for r in spark.sql(f"SHOW SCHEMAS IN `{mvm1_deploy_cat}`").collect()]
                except Exception:
                    re_schemas = []
            re_real = [s for s in re_schemas if not s.startswith("_") and s not in ("default", "information_schema")]
            print(f"  Reinstall succeeded. Schemas: {len(re_real)}")
            cleanup_td = _build_uninstall(6, "1", "mvm", mvm1_deploy_cat, test_id="06c_reinst", install_params=mvm1_install_params)
            run_test(cleanup_td)
            print("  Cleanup: uninstalled reinstalled model")
    else:
        R.append(_skip("06c_reinstall", "Reinstall After Uninstall (MVM v1)", "uninstall MVM v1 (test 06) did not pass"))

    # 07: Install MVM v2
    _banner("TEST 07: Install Model (MVM v2)", char="▓")
    mvm2_deploy_cat = ""
    mvm2_model_json = ""
    mvm2_install_params = {}
    if vibe_ok:
        td = _build_install(7, "2", "mvm", "mvm_v2_installed")
        mvm2_deploy_cat = td["_deploy_catalog"]
        mvm2_model_json = td["_model_json"]
        mvm2_install_params = dict(td["params"])
        R.append(run_test(td))
    else:
        R.append(_skip("07_install_mvm_v2", "Install Model (MVM v2)", "vibe modeling (test 01) failed"))
    mvm2_installed = R[-1].status == "PASSED"

    # 07b: Clash Detection — reinstall MVM v2 with SAME conventions as test 09
    _banner("TEST 07b: Clash Detection (reinstall MVM v2)", char="▓")
    if mvm2_installed and mvm2_model_json:
        td_clash2 = {
            "test_name": "07b_clash_detection_mvm_v2",
            "test_label": "Clash Detection (reinstall MVM v2)",
            "params": dict(mvm2_install_params),
        }
        td_clash2["params"]["deployment_catalog"] = mvm2_deploy_cat
        td_clash2["params"]["context_file"] = mvm2_model_json
        td_clash2["params"]["vibe_session_id"] = _gen_session_id()
        R.append(run_test_expect_clash(td_clash2))
    else:
        R.append(_skip("07b_clash_detection_mvm_v2", "Clash Detection (reinstall MVM v2)", "install MVM v2 (test 07) failed"))

    # 07c: Verify Install V2 Physical — schemas/tables for vibed model
    _banner("TEST 07c: Verify Install V2 Physical (MVM v2)", char="▓")
    if mvm2_installed:
        start_07c = time.time()
        deploy_07c = mvm2_deploy_cat
        issues_07c = []
        cat_style_07c = mvm2_install_params.get("cataloging_style", "One Catalog")
        cat_prefix_07c = mvm2_install_params.get("catalog_prefix", "")
        cat_suffix_07c = mvm2_install_params.get("catalog_suffix", "")

        all_schemas_07c, total_tables_07c, cats_checked_07c = _collect_schemas_and_tables(
            deploy_07c, cat_style_07c, cat_prefix_07c, cat_suffix_07c
        )

        if not all_schemas_09c:
            issues_07c.append(f"No user schemas found (style={cat_style_09c}, prefix='{cat_prefix_09c}', suffix='{cat_suffix_09c}', catalogs checked: {cats_checked_09c})")
        if total_tables_07c == 0 and all_schemas_09c:
            issues_07c.append(f"Found {len(all_schemas_07c)} schemas but 0 tables — v2 physical DDL may have failed")

        v1_tables_count_07c = 0
        if mvm1_deploy_cat and mvm1_install_params:
            _, v1_tables_count_07c, _ = _collect_schemas_and_tables(
                mvm1_deploy_cat,
                mvm1_install_params.get("cataloging_style", "One Catalog"),
                mvm1_install_params.get("catalog_prefix", ""),
                mvm1_install_params.get("catalog_suffix", ""),
            )

        dur_07c = time.time() - start_07c
        print(f"  V2 deploy catalog: {deploy_09c}")
        print(f"  V2 schemas: {len(all_schemas_07c)}, V2 tables: {total_tables_09c}")
        if v1_tables_count_09c:
            delta = total_tables_07c - v1_tables_count_07c
            print(f"  V1→V2 table delta: {'+' if delta >= 0 else ''}{delta} (vibes {'added' if delta > 0 else 'removed' if delta < 0 else 'did not change'} tables)")
        if issues_09c:
            for i in issues_09c:
                print(f"  ISSUE: {i}")
            R.append(TestResult("07c_verify_v2", "Verify Install V2 Physical (MVM v2)", "FAILED", dur_07c, "\n".join(issues_07c), mvm2_install_params))
        else:
            R.append(TestResult("07c_verify_v2", "Verify Install V2 Physical (MVM v2)", "PASSED", dur_07c, params=mvm2_install_params))
    else:
        R.append(_skip("07c_verify_v2", "Verify Install V2 Physical (MVM v2)", "install MVM v2 (test 07) failed"))

    # 08: Generate Samples on installed MVM v2
    _banner("TEST 08: Generate Samples (MVM v2)", char="▓")
    if mvm2_installed:
        R.append(run_test(_build_gen_samples(8, "2", "mvm", mvm2_deploy_cat, mvm2_model_json)))
    else:
        R.append(_skip("08_generate_samples_mvm_v2", "Generate Samples (MVM v2)", "install MVM v2 (test 07) failed"))

    # 09: Uninstall MVM v2
    _banner("TEST 09: Uninstall Model (MVM v2)", char="▓")
    if mvm2_installed:
        R.append(run_test(_build_uninstall(9, "2", "mvm", mvm2_deploy_cat, install_params=mvm2_install_params)))
    else:
        R.append(_skip("09_uninstall_mvm_v2", "Uninstall Model (MVM v2)", "install MVM v2 (test 07) failed"))

    # ─────────────────────────────────────────────────────────────
    # EDGE CASES (12–13): boundary conditions and alternate paths
    # ─────────────────────────────────────────────────────────────

    # 10: Empty Vibes — graceful exit (NEGATIVE TEST: expected to fail or exit gracefully)
    _banner("TEST 10: Empty Vibes Graceful Exit", char="▓")
    if mvm_ok:
        convs_10 = _random_conventions("10_emptyvibe")
        cat_w_10 = _random_catalog_widgets("10_emptyvibe")
        td_10 = {
            "test_name": "10_empty_vibes",
            "test_label": "Empty Vibes Graceful Exit",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "vibe modeling of version",
                "model_version": "1",
                "data_model_scopes": "Minimum Viable Model - MVM",
                "business_domains": "",
                "org_divisions": _pick("org_divisions_mvm"),
                "model_vibes": "",
                "deployment_catalog": w_catalog,
                "generate_samples": "0",
                "context_file": "",
                "vibe_session_id": _gen_session_id(),
                **convs_10,
                **cat_w_10,
            },
        }
        r12 = run_test(td_10)
        if r12.status == "PASSED":
            print("  Empty vibes: notebook exited cleanly (exit_with_warning path)")
            R.append(r12)
        elif r12.status == "FAILED":
            print(f"  ✅ Empty vibes: agent correctly rejected empty vibes (failed as expected)")
            R.append(TestResult("10_empty_vibes", "Empty Vibes Graceful Exit", "PASSED", r12.duration_seconds,
                f"Negative test passed (agent rejected empty vibes): {r12.error_msg[:300]}", td_10["params"]))
        else:
            R.append(r12)
    else:
        R.append(_skip("10_empty_vibes", "Empty Vibes Graceful Exit", "vibe runner (test 00) failed — no MVM base model"))

    # 11: Context-File Install — production workflow
    _banner("TEST 11: Context-File Install (MVM v1)", char="▓")
    if mvm_ok:
        ctx_model_json = _find_model_json("1", "mvm")
        ctx_deploy_cat = f"{w_catalog}_{sanitized_name}_ctx_install"
        convs_11 = _random_conventions("11_ctxinst")
        cat_w_11 = _random_catalog_widgets("11_ctxinst")
        td_11 = {
            "test_name": "11_ctx_install",
            "test_label": "Context-File Install (MVM v1)",
            "params": {
                "business_name": w_business_name,
                "business_description": w_business_description,
                "operation": "install model",
                "model_version": "1",
                "data_model_scopes": "Minimum Viable Model - MVM",
                "business_domains": "",
                "org_divisions": "Operations and Business",
                "model_vibes": "",
                "deployment_catalog": ctx_deploy_cat,
                "generate_samples": "0",
                "context_file": ctx_model_json,
                "vibe_session_id": _gen_session_id(),
                **convs_11,
                **cat_w_11,
            },
        }
        R.append(run_test(td_11))
        if R[-1].status == "PASSED":
            try:
                ctx_schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN `{ctx_deploy_cat}`").collect()]
            except Exception:
                try:
                    ctx_schemas = [r.namespace for r in spark.sql(f"SHOW SCHEMAS IN `{ctx_deploy_cat}`").collect()]
                except Exception:
                    ctx_schemas = []
            ctx_real = [s for s in ctx_schemas if not s.startswith("_") and s not in ("default", "information_schema")]
            ctx_table_cnt = 0
            for s in ctx_real[:20]:
                try:
                    ctx_table_cnt += len(spark.sql(f"SHOW TABLES IN `{ctx_deploy_cat}`.`{s}`").collect())
                except Exception:
                    pass
            print(f"  Context-file install succeeded.")
            print(f"  Deploy catalog: {ctx_deploy_cat}")
            print(f"  Schemas: {len(ctx_real)}, Tables: {ctx_table_cnt}")
            cleanup_11 = _build_uninstall(11, "1", "mvm", ctx_deploy_cat, test_id="11_ctxclean", install_params=td_11["params"])
            run_test(cleanup_11)
            try:
                spark.sql(f"DROP CATALOG IF EXISTS `{ctx_deploy_cat}` CASCADE")
            except Exception:
                pass
            print("  Cleanup: uninstalled and dropped context-install catalog")
    else:
        R.append(_skip("11_ctx_install", "Context-File Install (MVM v1)", "vibe runner (test 00) failed — no MVM base model"))

    # ─────────────────────────────────────────────────────────────
    # CROSS-CONVENTION INSTALL TEST (13c): install with DIFFERENT
    # physical layout and conventions than what was generated
    # ─────────────────────────────────────────────────────────────

    _banner("TEST 11c: Cross-Convention Install (MVM v1 → different layout)", char="▓")
    if mvm_ok:
        xconv_model_json = _find_model_json("1", "mvm")
        xconv_deploy_cat = f"{w_catalog}_{sanitized_name}_xconv_install"
        xconv_params = {
            "business_name": w_business_name,
            "business_description": w_business_description,
            "operation": "install model",
            "model_version": "1",
            "data_model_scopes": "Minimum Viable Model - MVM",
            "business_domains": "",
            "org_divisions": "Operations and Business",
            "model_vibes": "",
            "deployment_catalog": xconv_deploy_cat,
            "generate_samples": "0",
            "context_file": xconv_model_json,
            "vibe_session_id": _gen_session_id(),
            "naming_convention": "PascalCase",
            "primary_key_suffix": "_key",
            "schema_prefix": "dw_",
            "schema_suffix": "_zone",
            "tag_prefix": "meta_",
            "tag_suffix": "_v1",
            "table_id_type": "INT",
            "boolean_format": "String (Y/N)",
            "date_format": "dd/MM/yyyy",
            "timestamp_format": "yyyy-MM-dd'T'HH:mm:ss",
            "classification_levels": "top_secret=Top Secret, secret=Secret, classified=Classified, unclassified=Unclassified",
            "housekeeping_columns": "Yes",
            "history_tracking_columns": "Yes",
            "cataloging_style": "Catalog per Division",
            "catalog_prefix": "t11c_xconv_",
            "catalog_suffix": "",
        }
        td_11c = {
            "test_name": "11c_xconv_install",
            "test_label": "Cross-Convention Install (MVM v1 → different layout)",
            "params": xconv_params,
        }
        R.append(run_test(td_11c))
        xconv_installed = R[-1].status == "PASSED"

        if xconv_installed:
            _banner("TEST 11d: Verify Cross-Convention Physical Layout", char="▓")
            start_11d = time.time()
            issues_11d = []

            xconv_schemas, xconv_tables, xconv_cats = _collect_schemas_and_tables(
                xconv_deploy_cat, "Catalog per Division", "t11c_xconv_", ""
            )

            if not xconv_schemas:
                issues_11d.append(f"No schemas found (checked catalogs: {xconv_cats})")
            if xconv_tables == 0 and xconv_schemas:
                issues_11d.append(f"Found {len(xconv_schemas)} schemas but 0 tables")

            division_cats = _find_domain_catalogs("t11c_xconv_", "")
            if not division_cats:
                issues_11d.append("Catalog per Division: no catalogs found with prefix 't11c_xconv_'")
            else:
                print(f"  Division catalogs found: {len(division_cats)} → {division_cats[:5]}")

            xconv_mm = f"`{w_catalog}`.`_metamodel`"
            xconv_nc_pass = 0
            xconv_nc_total = 0
            xconv_pk_pass = 0
            xconv_pk_total = 0
            xconv_schema_pass = 0
            xconv_schema_total = 0

            try:
                xconv_prods = spark.sql(
                    f"SELECT product, table_name, primary_key FROM {xconv_mm}.product "
                    f"WHERE LOWER(business) = LOWER('{w_business_name}') AND version = '1' "
                    f"AND (model_scope = 'mvm' OR model_scope IS NULL)"
                ).collect()
                for row in xconv_prods:
                    tn = row.table_name or ""
                    pk = row.primary_key or ""
                    if tn:
                        xconv_nc_total += 1
                        if tn[0].isupper() and "_" not in tn:
                            xconv_nc_pass += 1
                    if pk:
                        xconv_pk_total += 1
                        if pk.endswith("_key") or pk.lower().endswith("_key"):
                            xconv_pk_pass += 1
            except Exception as e:
                issues_11d.append(f"Cannot query metamodel for convention check: {e}")

            try:
                xconv_doms = spark.sql(
                    f"SELECT domain, database_name FROM {xconv_mm}.domain "
                    f"WHERE LOWER(business) = LOWER('{w_business_name}') AND version = '1' "
                    f"AND (model_scope = 'mvm' OR model_scope IS NULL)"
                ).collect()
                for row in xconv_doms:
                    db = row.database_name or ""
                    if db:
                        xconv_schema_total += 1
                        if db.lower().startswith("dw_"):
                            xconv_schema_pass += 1
            except Exception:
                pass

            xconv_sfx_total = 0
            xconv_sfx_pass = 0
            try:
                _xc_physical = [s.split(".", 1)[1] if "." in s else s for s in xconv_schemas]
                for pn in _xc_physical:
                    xconv_sfx_total += 1
                    if pn.lower().endswith("_zone"):
                        xconv_sfx_pass += 1
                xconv_schema_total += xconv_sfx_total
                xconv_schema_pass += xconv_sfx_pass
            except Exception:
                pass

            print(f"  Schemas: {len(xconv_schemas)}, Tables: {xconv_tables}")
            nc_pct = round(100 * xconv_nc_pass / xconv_nc_total, 1) if xconv_nc_total else 0
            pk_pct = round(100 * xconv_pk_pass / xconv_pk_total, 1) if xconv_pk_total else 0
            sch_pct = round(100 * xconv_schema_pass / xconv_schema_total, 1) if xconv_schema_total else 0
            print(f"  PascalCase compliance: {xconv_nc_pass}/{xconv_nc_total} ({nc_pct}%)")
            print(f"  PK suffix '_key' compliance: {xconv_pk_pass}/{xconv_pk_total} ({pk_pct}%)")
            print(f"  Schema prefix 'dw_' (metamodel) + suffix '_zone' (physical): {xconv_schema_pass}/{xconv_schema_total} ({sch_pct}%)")

            if nc_pct < 70:
                issues_11d.append(f"PascalCase compliance too low: {nc_pct}%")
            if pk_pct < 70:
                issues_11d.append(f"PK suffix '_key' compliance too low: {pk_pct}%")
            if sch_pct < 70:
                issues_11d.append(f"Schema naming compliance too low: {sch_pct}%")

            dur_11d = time.time() - start_11d
            if issues_13d:
                for iss in issues_13d:
                    print(f"  ISSUE: {iss}")
                R.append(TestResult("11d_xconv_verify", "Verify Cross-Convention Physical Layout", "FAILED", dur_11d,
                    "\n".join(issues_11d), xconv_params))
            else:
                R.append(TestResult("11d_xconv_verify", "Verify Cross-Convention Physical Layout", "PASSED", dur_11d,
                    params=xconv_params))

            cleanup_11c = _build_uninstall(11, "1", "mvm", xconv_deploy_cat, test_id="11c_xclean", install_params=xconv_params)
            run_test(cleanup_11c)
            for dc in division_cats:
                try:
                    spark.sql(f"DROP CATALOG IF EXISTS `{dc}` CASCADE")
                except Exception:
                    pass
            try:
                spark.sql(f"DROP CATALOG IF EXISTS `{xconv_deploy_cat}` CASCADE")
            except Exception:
                pass
            print("  Cleanup: uninstalled and dropped cross-convention catalogs")
        else:
            R.append(_skip("11d_xconv_verify", "Verify Cross-Convention Physical Layout",
                "cross-convention install (test 11c) failed"))
    else:
        R.append(_skip("11c_xconv_install", "Cross-Convention Install (MVM v1 → different layout)", "vibe runner (test 00) failed — no MVM base model"))
        R.append(_skip("11d_xconv_verify", "Verify Cross-Convention Physical Layout", "vibe runner (test 00) failed — no MVM base model"))

    # ─────────────────────────────────────────────────────────────
    # NEGATIVE / GUARD-RAIL TESTS (14–15): verify error quality
    # ─────────────────────────────────────────────────────────────

    # 12: Invalid Context File — guard-rail test
    _banner("TEST 12: Invalid Context File Guard-Rails", char="▓")
    start_12 = time.time()
    sub_results_12 = []

    bogus_path = "/Volumes/fake_catalog/fake_schema/fake_volume/nonexistent_model.json"
    convs_12 = _random_conventions("12_badctx")
    cat_w_12 = _random_catalog_widgets("12_badctx")
    td_14_bogus = {
        "test_name": "12_invalid_ctx",
        "test_label": "Invalid Context File Guard-Rails",
        "params": {
            "business_name": w_business_name,
            "business_description": w_business_description,
            "operation": "install model",
            "model_version": "1",
            "data_model_scopes": "Minimum Viable Model - MVM",
            "business_domains": "",
            "org_divisions": "Operations and Business",
            "model_vibes": "",
            "deployment_catalog": f"{w_catalog}_neg_bogus",
            "generate_samples": "0",
            "context_file": bogus_path,
            "vibe_session_id": _gen_session_id(),
            **convs_12,
            **cat_w_12,
        },
    }
    r14a = run_test(td_14_bogus)
    if r14a.status == "PASSED":
        sub_results_12.append("FAIL: non-existent context_file did NOT cause an error — notebook should reject it")
        try:
            spark.sql(f"DROP CATALOG IF EXISTS `{w_catalog}_neg_bogus` CASCADE")
        except Exception:
            pass
    else:
        err_lower_14a = (r14a.error_msg or "").lower()
        if any(kw in err_lower_14a for kw in ["not found", "does not exist", "no such file", "cannot read", "filenotfound", "path", "context_file"]):
            sub_results_12.append("PASS: non-existent path rejected with clear message")
        else:
            sub_results_12.append(f"PASS: non-existent path rejected (agent failed as expected): {r14a.error_msg[:200]}")

    pasted_json = '{"business": "Bank", "domains": [{"name": "ops"}]}'
    td_14_pasted = {
        "test_name": "12_invalid_ctx_pasted",
        "test_label": "Invalid Context File (pasted JSON)",
        "params": dict(td_14_bogus["params"]),
    }
    td_14_pasted["params"]["context_file"] = pasted_json
    td_14_pasted["params"]["deployment_catalog"] = f"{w_catalog}_neg_pasted"
    td_14_pasted["params"]["vibe_session_id"] = _gen_session_id()
    r14b = run_test(td_14_pasted)
    if r14b.status == "PASSED":
        sub_results_12.append("FAIL: pasted JSON in context_file did NOT cause an error — should be rejected")
        try:
            spark.sql(f"DROP CATALOG IF EXISTS `{w_catalog}_neg_pasted` CASCADE")
        except Exception:
            pass
    else:
        err_lower_14b = (r14b.error_msg or "").lower()
        if any(kw in err_lower_14b for kw in ["not a valid path", "must be a path", "looks like json", "pasted", "context_file"]):
            sub_results_12.append("PASS: pasted JSON rejected with clear message")
        else:
            sub_results_12.append(f"PASS: pasted JSON rejected (agent failed as expected): {r14b.error_msg[:200]}")

    dur_12 = time.time() - start_12
    fails_12 = sum(1 for r in sub_results_12 if r.startswith("FAIL"))
    for r in sub_results_14:
        print(f"  {r}")
    if fails_12 == 0:
        R.append(TestResult("12_invalid_ctx", "Invalid Context File Guard-Rails", "PASSED", dur_12, params=td_14_bogus["params"]))
    else:
        R.append(TestResult("12_invalid_ctx", "Invalid Context File Guard-Rails", "FAILED", dur_12,
            "\n".join(r for r in sub_results_12 if r.startswith("FAIL")), td_14_bogus["params"]))

    # 13: Missing Business Name — guard-rail test
    _banner("TEST 13: Missing Business Name Guard-Rail", char="▓")
    convs_13 = _random_conventions("13_noname")
    cat_w_13 = _random_catalog_widgets("13_noname")
    td_13 = {
        "test_name": "13_no_biz_name",
        "test_label": "Missing Business Name Guard-Rail",
        "params": {
            "business_name": "",
            "business_description": "test with no name",
            "operation": "new base model",
            "model_version": "",
            "data_model_scopes": "Minimum Viable Model - MVM",
            "business_domains": "",
            "org_divisions": "Operations and Business",
            "model_vibes": "",
            "deployment_catalog": w_catalog,
            "generate_samples": "0",
            "context_file": "",
            "vibe_session_id": _gen_session_id(),
            **convs_13,
            **cat_w_13,
        },
    }
    r15 = run_test(td_13)
    if r15.status == "PASSED":
        R.append(TestResult("13_no_biz_name", "Missing Business Name Guard-Rail", "FAILED", r15.duration_seconds,
            "Empty business_name was accepted — validation guard is broken!", td_13["params"]))
    else:
        err_lower_13 = (r15.error_msg or "").lower()
        clear_msg = any(kw in err_lower_13 for kw in [
            "business_name", "business name", "required", "cannot be empty",
            "must provide", "missing", "blank",
        ])
        print(f"  Empty business_name correctly rejected.")
        print(f"  Error clarity: {'clear' if clear_msg else 'VAGUE — needs improvement'}")
        print(f"  Error excerpt: {r15.error_msg[:300]}")
        R.append(TestResult("13_no_biz_name", "Missing Business Name Guard-Rail", "PASSED", r15.duration_seconds, params=td_13["params"]))
        if not clear_msg:
            print("  NOTE: Error message doesn't explicitly mention 'business_name' — consider improving error text")

    # ═════════════════════════════════════════════════════════════
    # PHASE 5: MERGE LOGS
    # ═════════════════════════════════════════════════════════════

    passed, failed, skipped, warned = merge_all_logs(R)
    total_duration = time.time() - overall_start

    # ═════════════════════════════════════════════════════════════
    # PHASE 6: VERIFICATION, AUDIT & QUALITY REPORT
    # ═════════════════════════════════════════════════════════════

    _banner("PHASE 6: VERIFICATION & QUALITY AUDIT")

    def _check_naming(name, convention):
        if not name or not convention:
            return True
        s = name.strip()
        if not s:
            return True
        if convention == "snake_case":
            return s == s.lower()
        elif convention == "camelCase":
            return s[0].islower() and "_" not in s
        elif convention == "PascalCase":
            return s[0].isupper() and "_" not in s
        elif convention == "SCREAMING_CASE":
            return s == s.upper()
        return True

    audit_results = {}

    def verify_conventions(test_result, version, scope):
        """Query the metamodel and verify that the requested conventions were actually applied."""
        params = test_result.params
        nc = params.get("naming_convention", "")
        pk_suffix = params.get("primary_key_suffix", "")
        schema_prefix = params.get("schema_prefix", "")
        schema_suffix = params.get("schema_suffix", "")
        tag_prefix = params.get("tag_prefix", "")
        tag_suffix = params.get("tag_suffix", "")
        id_type = params.get("table_id_type", "")

        mm = f"`{w_catalog}`.`_metamodel`"
        scope_sql = f" AND (model_scope = '{scope}' OR model_scope IS NULL)"
        checks = []
        totals = {"columns_checked": 0, "columns_pass": 0, "pk_checked": 0, "pk_pass": 0,
                  "schema_checked": 0, "schema_pass": 0, "tag_checked": 0, "tag_pass": 0,
                  "id_type_checked": 0, "id_type_pass": 0}

        try:
            attr_rows = spark.sql(
                f"SELECT column_name, tags, type, foreign_key_to FROM {mm}.attribute "
                f"WHERE LOWER(business) = LOWER('{w_business_name}') AND version = '{version}'{scope_sql}"
            ).collect()

            prod_rows = spark.sql(
                f"SELECT product, table_name, primary_key FROM {mm}.product "
                f"WHERE LOWER(business) = LOWER('{w_business_name}') AND version = '{version}'{scope_sql}"
            ).collect()

            domain_rows = spark.sql(
                f"SELECT domain, database_name FROM {mm}.domain "
                f"WHERE LOWER(business) = LOWER('{w_business_name}') AND version = '{version}'{scope_sql}"
            ).collect()
        except Exception as e:
            checks.append(f"⚠️  Could not query metamodel for verification: {str(e)[:200]}")
            audit_results[test_result.test_name] = {"score": -1, "checks": checks, "totals": totals}
            return

        if nc:
            for row in attr_rows:
                cn = row.column_name or ""
                if not cn:
                    continue
                totals["columns_checked"] += 1
                if _check_naming(cn, nc):
                    totals["columns_pass"] += 1

            for row in prod_rows:
                tn = row.table_name or ""
                if not tn:
                    continue
                totals["columns_checked"] += 1
                if _check_naming(tn, nc):
                    totals["columns_pass"] += 1

            if totals["columns_checked"] > 0:
                pct = round(100 * totals["columns_pass"] / totals["columns_checked"], 1)
                icon = "✅" if pct >= 90 else ("⚠️" if pct >= 70 else "❌")
                checks.append(f"{icon} Naming convention '{nc}': {totals['columns_pass']}/{totals['columns_checked']} comply ({pct}%)")
            else:
                checks.append(f"⚠️  No columns found to verify naming convention")

        if pk_suffix:
            for row in prod_rows:
                pk = row.primary_key or ""
                if not pk:
                    continue
                totals["pk_checked"] += 1
                if pk.endswith(pk_suffix) or pk.lower().endswith(pk_suffix.lower()):
                    totals["pk_pass"] += 1

            if totals["pk_checked"] > 0:
                pct = round(100 * totals["pk_pass"] / totals["pk_checked"], 1)
                icon = "✅" if pct >= 90 else ("⚠️" if pct >= 70 else "❌")
                checks.append(f"{icon} PK suffix '{pk_suffix}': {totals['pk_pass']}/{totals['pk_checked']} comply ({pct}%)")

        if schema_prefix or schema_suffix:
            if schema_prefix:
                for row in domain_rows:
                    db = row.database_name or ""
                    if not db:
                        continue
                    totals["schema_checked"] += 1
                    if db.lower().startswith(schema_prefix.lower()):
                        totals["schema_pass"] += 1

            if schema_suffix:
                _vc_deploy_cat = params.get("deployment_catalog", w_catalog)
                _vc_cat_style = params.get("cataloging_style", "One Catalog")
                _vc_cat_prefix = params.get("catalog_prefix", "")
                _vc_cat_suffix = params.get("catalog_suffix", "")
                try:
                    _vc_schemas, _, _vc_cats = _collect_schemas_and_tables(
                        _vc_deploy_cat, _vc_cat_style, _vc_cat_prefix, _vc_cat_suffix
                    )
                    _vc_physical_names = [s.split(".", 1)[1] if "." in s else s for s in _vc_schemas]
                    for pn in _vc_physical_names:
                        totals["schema_checked"] += 1
                        if pn.lower().endswith(schema_suffix.lower()):
                            totals["schema_pass"] += 1
                except Exception:
                    pass

            if totals["schema_checked"] > 0:
                pct = round(100 * totals["schema_pass"] / totals["schema_checked"], 1)
                icon = "✅" if pct >= 90 else ("⚠️" if pct >= 70 else "❌")
                pfx_sfx = f"prefix='{schema_prefix}' suffix='{schema_suffix}'"
                checks.append(f"{icon} Schema naming ({pfx_sfx}): {totals['schema_pass']}/{totals['schema_checked']} comply ({pct}%)")

        if tag_prefix or tag_suffix:
            for row in attr_rows:
                tags_str = row.tags or ""
                if not tags_str.strip():
                    continue
                for tag in tags_str.split(","):
                    tag = tag.strip()
                    if not tag:
                        continue
                    totals["tag_checked"] += 1
                    ok = True
                    if tag_prefix and not tag.lower().startswith(tag_prefix.lower()):
                        ok = False
                    if tag_suffix and not tag.lower().endswith(tag_suffix.lower()):
                        ok = False
                    if ok:
                        totals["tag_pass"] += 1

            if totals["tag_checked"] > 0:
                pct = round(100 * totals["tag_pass"] / totals["tag_checked"], 1)
                icon = "✅" if pct >= 90 else ("⚠️" if pct >= 70 else "❌")
                checks.append(f"{icon} Tag naming (prefix='{tag_prefix}' suffix='{tag_suffix}'): {totals['tag_pass']}/{totals['tag_checked']} comply ({pct}%)")

        if id_type:
            expected_types = {id_type.upper()}
            if id_type.upper() == "LONG":
                expected_types.add("BIGINT")
            elif id_type.upper() == "BIGINT":
                expected_types.add("LONG")
            for row in attr_rows:
                tags_str = (row.tags or "").lower()
                if "primary_key" not in tags_str:
                    continue
                totals["id_type_checked"] += 1
                attr_type = (row.type or "").upper().strip()
                if attr_type in expected_types:
                    totals["id_type_pass"] += 1

            if totals["id_type_checked"] > 0:
                pct = round(100 * totals["id_type_pass"] / totals["id_type_checked"], 1)
                icon = "✅" if pct >= 90 else ("⚠️" if pct >= 70 else "❌")
                checks.append(f"{icon} PK ID type '{id_type}': {totals['id_type_pass']}/{totals['id_type_checked']} comply ({pct}%)")

        total_c = sum(v for k, v in totals.items() if k.endswith("_checked"))
        total_p = sum(v for k, v in totals.items() if k.endswith("_pass"))
        score = round(100 * total_p / total_c, 1) if total_c > 0 else 100.0
        checks.append(f"{'─'*40}")
        checks.append(f"📊 Convention compliance score: {score}% ({total_p}/{total_c})")

        model_stats = {"domains": len(domain_rows), "products": len(prod_rows), "attributes": len(attr_rows)}
        fk_count = sum(1 for r in attr_rows if (r.foreign_key_to or "").strip())
        model_stats["foreign_keys"] = fk_count
        checks.append(f"📈 Model stats: {model_stats['domains']} domains, {model_stats['products']} products, {model_stats['attributes']} attributes, {model_stats['foreign_keys']} FKs")

        audit_results[test_result.test_name] = {"score": score, "checks": checks, "totals": totals, "model_stats": model_stats}

    def audit_vibe_effectiveness(base_version, new_version, scope, vibes_text):
        """Compare two model versions to measure vibe modeling impact."""
        mm = f"`{w_catalog}`.`_metamodel`"
        scope_sql = f" AND (model_scope = '{scope}' OR model_scope IS NULL)"
        biz_lower = w_business_name.lower()
        result = {"delta": {}, "vibes_followed": [], "vibes_missed": [], "score": 0}

        try:
            def _count(table, ver):
                rows = spark.sql(
                    f"SELECT COUNT(*) as cnt FROM {mm}.{table} "
                    f"WHERE LOWER(business) = LOWER('{biz_lower}') AND version = '{ver}'{scope_sql}"
                ).collect()
                return rows[0].cnt if rows else 0

            def _domains(ver):
                rows = spark.sql(
                    f"SELECT domain FROM {mm}.domain "
                    f"WHERE LOWER(business) = LOWER('{biz_lower}') AND version = '{ver}'{scope_sql}"
                ).collect()
                return {r.domain.lower() for r in rows if r.domain}

            def _products(ver):
                rows = spark.sql(
                    f"SELECT domain, product FROM {mm}.product "
                    f"WHERE LOWER(business) = LOWER('{biz_lower}') AND version = '{ver}'{scope_sql}"
                ).collect()
                return {f"{r.domain}.{r.product}".lower() for r in rows if r.product}

            v1_d = _count("domain", base_version)
            v2_d = _count("domain", new_version)
            v1_p = _count("product", base_version)
            v2_p = _count("product", new_version)
            v1_a = _count("attribute", base_version)
            v2_a = _count("attribute", new_version)

            result["delta"] = {
                "domains": f"{v1_d} → {v2_d} ({'+' if v2_d >= v1_d else ''}{v2_d - v1_d})",
                "products": f"{v1_p} → {v2_p} ({'+' if v2_p >= v1_p else ''}{v2_p - v1_p})",
                "attributes": f"{v1_a} → {v2_a} ({'+' if v2_a >= v1_a else ''}{v2_a - v1_a})",
            }

            v1_domains = _domains(base_version)
            v2_domains = _domains(new_version)
            v1_products = _products(base_version)
            v2_products = _products(new_version)

            new_domains = v2_domains - v1_domains
            removed_domains = v1_domains - v2_domains
            new_products = v2_products - v1_products
            result["delta"]["new_domains"] = sorted(new_domains) if new_domains else []
            result["delta"]["removed_domains"] = sorted(removed_domains) if removed_domains else []
            result["delta"]["new_products"] = sorted(list(new_products)[:20]) if new_products else []

            vibes_lower = (vibes_text or "").lower()
            score = 50
            if "add" in vibes_lower and "domain" in vibes_lower:
                if new_domains:
                    result["vibes_followed"].append(f"✅ New domain(s) added: {', '.join(sorted(new_domains)[:5])}")
                    score += 15
                else:
                    result["vibes_missed"].append(f"❌ Vibes requested new domain but none were added")
                    score -= 10

            if "rename" in vibes_lower and "domain" in vibes_lower:
                if removed_domains:
                    result["vibes_followed"].append(f"✅ Domain(s) renamed/removed: {', '.join(sorted(removed_domains)[:5])}")
                    score += 15
                else:
                    result["vibes_missed"].append(f"⚠️  Vibes requested domain rename but no domains were removed/renamed")

            if "add" in vibes_lower and "product" in vibes_lower:
                if new_products:
                    result["vibes_followed"].append(f"✅ New product(s) added: {len(new_products)}")
                    score += 15
                else:
                    result["vibes_missed"].append(f"❌ Vibes requested new product but none were added")
                    score -= 10

            if v2_p > v1_p:
                result["vibes_followed"].append(f"✅ Model grew: {v1_p} → {v2_p} products (+{v2_p - v1_p})")
                score += 5
            elif v2_p == v1_p:
                result["vibes_missed"].append(f"⚠️  Product count unchanged ({v1_p})")
            elif v2_p < v1_p:
                result["vibes_missed"].append(f"⚠️  Product count decreased: {v1_p} → {v2_p}")

            result["score"] = max(0, min(100, score))
        except Exception as e:
            result["error"] = str(e)[:300]
            result["score"] = -1

        audit_results["vibe_audit"] = result
        return result

    _MODEL_PRODUCING_OPS = {"new base model", "vibe modeling of version", "shrink ecm", "enlarge mvm"}

    def _find_latest_version_in_metamodel(target_scope):
        mm = f"`{w_catalog}`.`_metamodel`"
        try:
            rows = spark.sql(
                f"SELECT DISTINCT version FROM {mm}.business "
                f"WHERE LOWER(business) = LOWER('{w_business_name}') "
                f"AND (model_scope = '{target_scope}' OR model_scope IS NULL) "
                f"ORDER BY version DESC"
            ).collect()
            if rows:
                return rows[0].version
        except Exception:
            pass
        return None

    for tr in R:
        if tr.status != "PASSED":
            continue
        op = (tr.params.get("operation") or "").strip().lower()
        if op not in _MODEL_PRODUCING_OPS:
            continue
        version = tr.params.get("model_version", "1") or "1"
        scope_raw = (tr.params.get("data_model_scopes") or "").lower()
        scope = "ecm" if "ecm" in scope_raw else "mvm"
        if op == "new base model":
            ver_to_check = "1"
        elif op == "vibe modeling of version":
            try:
                ver_to_check = str(int(float(version)) + 1)
            except (ValueError, TypeError):
                ver_to_check = "2"
            scope = "mvm"
        elif op == "shrink ecm":
            scope = "mvm"
            ver_to_check = _find_latest_version_in_metamodel(scope) or version
        elif op == "enlarge mvm":
            scope = "ecm"
            ver_to_check = _find_latest_version_in_metamodel(scope) or version
        else:
            ver_to_check = version

        print(f"\n  🔍 Verifying conventions for [{tr.test_name}] v{ver_to_check}_{scope}...")
        verify_conventions(tr, ver_to_check, scope)
        ar = audit_results.get(tr.test_name, {})
        for line in ar.get("checks", []):
            print(f"     {line}")

    if vibe_ok and mvm_ok:
        base_ver = "1"
        latest_mvm = _find_latest_version_in_metamodel("mvm")
        if latest_mvm and latest_mvm != base_ver:
            new_ver = latest_mvm
        else:
            try:
                new_ver = str(int(float(base_ver)) + 1)
            except (ValueError, TypeError):
                new_ver = "2"
        vibes_text = w_model_vibes
        print(f"\n  🔬 Auditing vibe effectiveness: v{base_ver}_mvm → v{new_ver}_mvm...")
        va = audit_vibe_effectiveness(base_ver, new_ver, "mvm", vibes_text)
        if va.get("error"):
            print(f"     ⚠️  Vibe audit error: {va['error']}")
        else:
            print(f"     📊 Vibe Effectiveness Score: {va['score']}%")
            for k, v in va.get("delta", {}).items():
                if isinstance(v, list):
                    if v:
                        print(f"     {k}: {', '.join(v[:5])}")
                else:
                    print(f"     {k}: {v}")
            for line in va.get("vibes_followed", []):
                print(f"     {line}")
            for line in va.get("vibes_missed", []):
                print(f"     {line}")

    # ═════════════════════════════════════════════════════════════
    # PHASE 7: FINAL QUALITY REPORT
    # ═════════════════════════════════════════════════════════════

    _banner("PHASE 7: FINAL QUALITY REPORT")

    conv_scores = [v["score"] for v in audit_results.values() if isinstance(v.get("score"), (int, float)) and v["score"] >= 0 and "checks" in v]
    avg_conv = round(sum(conv_scores) / len(conv_scores), 1) if conv_scores else -1
    vibe_score = audit_results.get("vibe_audit", {}).get("score", -1)
    test_pass_rate = round(100 * passed / (passed + failed), 1) if (passed + failed) > 0 else 0
    sample_gen_ok = sum(1 for tr in R if "generate_samples" in tr.test_name and tr.status == "PASSED")
    sample_gen_total = sum(1 for tr in R if "generate_samples" in tr.test_name and tr.status != "SKIPPED")

    overall_score = 0.0
    weight_count = 0
    if test_pass_rate >= 0:
        overall_score += test_pass_rate * 0.40
        weight_count += 1
    if avg_conv >= 0:
        overall_score += avg_conv * 0.30
        weight_count += 1
    if vibe_score >= 0:
        overall_score += vibe_score * 0.20
        weight_count += 1
    if sample_gen_total > 0:
        sample_pct = round(100 * sample_gen_ok / sample_gen_total, 1)
        overall_score += sample_pct * 0.10
        weight_count += 1
    else:
        sample_pct = -1

    if weight_count > 0:
        divisor = 0.40 + (0.30 if avg_conv >= 0 else 0) + (0.20 if vibe_score >= 0 else 0) + (0.10 if sample_pct >= 0 else 0)
        overall_score = round(overall_score / divisor, 1) if divisor > 0 else 0
    else:
        overall_score = 0

    if overall_score >= 90:
        sentiment = "🟢 EXCELLENT"
        sentiment_msg = "Pipeline is performing exceptionally well."
    elif overall_score >= 75:
        sentiment = "🟡 GOOD"
        sentiment_msg = "Pipeline works but has room for improvement."
    elif overall_score >= 50:
        sentiment = "🟠 NEEDS WORK"
        sentiment_msg = "Significant issues need attention before production use."
    else:
        sentiment = "🔴 POOR"
        sentiment_msg = "Critical failures detected. Do not use in production."

    improvements = []
    if test_pass_rate < 100:
        failed_tests = [tr.test_label for tr in R if tr.status == "FAILED"]
        improvements.append(f"Fix failing tests: {', '.join(failed_tests[:5])}")
    if avg_conv >= 0 and avg_conv < 90:
        low_conv = [(k, v["score"]) for k, v in audit_results.items() if "checks" in v and 0 <= v.get("score", -1) < 90]
        for name, sc in low_conv[:3]:
            improvements.append(f"Convention compliance low ({sc}%) for {name}")
    if vibe_score >= 0 and vibe_score < 70:
        for line in audit_results.get("vibe_audit", {}).get("vibes_missed", []):
            improvements.append(f"Vibe issue: {line}")
    if sample_pct >= 0 and sample_pct < 100:
        improvements.append(f"Sample generation: {sample_gen_ok}/{sample_gen_total} succeeded — fix PK/column mismatches")
    if skipped > 0:
        improvements.append(f"{skipped} test(s) skipped due to upstream failures — fix root cause first")

    for k, v in audit_results.items():
        if "checks" not in v:
            continue
        ms = v.get("model_stats", {})
        if ms.get("foreign_keys", 999) == 0 and ms.get("products", 0) > 1:
            improvements.append(f"{k}: 0 foreign keys detected — model has no relational integrity")
        if ms.get("attributes", 0) > 0 and ms.get("products", 0) > 0:
            avg_attrs = ms["attributes"] / ms["products"]
            if avg_attrs < 5:
                improvements.append(f"{k}: avg {avg_attrs:.0f} attributes/product — models are too shallow")

    report_lines = []
    report_lines.append(f"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║  📋 VIBE TESTER — QUALITY REPORT                                                ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  Overall Score:   {overall_score:>5.1f}%   {sentiment:<46}║
║  Sentiment:       {sentiment_msg:<60}║
║                                                                                  ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  BREAKDOWN                                                                       ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  Test Pass Rate:          {test_pass_rate:>5.1f}%  ({passed}/{passed+failed} tests)                        ║
║  Convention Compliance:   {avg_conv if avg_conv >= 0 else 'N/A':>5}{'%' if avg_conv >= 0 else ' '}  (avg across {len(conv_scores)} model-producing tests)          ║
║  Vibe Effectiveness:      {vibe_score if vibe_score >= 0 else 'N/A':>5}{'%' if vibe_score >= 0 else ' '}  (v1→v2 comparison)                              ║
║  Sample Generation:       {f'{sample_pct}%' if sample_pct >= 0 else 'N/A':>5}   ({sample_gen_ok}/{sample_gen_total} tests)                             ║
║                                                                                  ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  PER-TEST CONVENTION SCORES                                                      ║
╠══════════════════════════════════════════════════════════════════════════════════╣""")

    for tr in R:
        ar = audit_results.get(tr.test_name)
        if not ar or "checks" not in ar:
            continue
        sc = ar["score"]
        ms = ar.get("model_stats", {})
        icon = "✅" if sc >= 90 else ("⚠️" if sc >= 70 else "❌")
        nc = tr.params.get("naming_convention", "?")
        pk = tr.params.get("primary_key_suffix", "?")
        line = f"║  {icon} {tr.test_label[:40]:<40} {sc:>5.1f}%  nc={nc} pk={pk}  D={ms.get('domains',0)} P={ms.get('products',0)} A={ms.get('attributes',0)} FK={ms.get('foreign_keys',0)}"
        report_lines.append(f"{line:<81}║")

    report_lines.append(f"║{'':80}║")
    report_lines.append(f"╠══════════════════════════════════════════════════════════════════════════════════╣")
    report_lines.append(f"║  IMPROVEMENT RECOMMENDATIONS                                                   ║")
    report_lines.append(f"╠══════════════════════════════════════════════════════════════════════════════════╣")

    if improvements:
        for i, imp in enumerate(improvements[:10], 1):
            line = f"║  {i:>2}. {imp[:73]}"
            report_lines.append(f"{line:<81}║")
    else:
        report_lines.append(f"║  🎉 No issues found — pipeline is performing optimally!                        ║")

    report_lines.append(f"║{'':80}║")
    report_lines.append(f"╚══════════════════════════════════════════════════════════════════════════════════╝")

    full_report = "\n".join(report_lines)
    print(full_report)

    try:
        dbutils.fs.put(f"{log_output_dir}/quality_report.log", full_report, overwrite=True)
        print(f"\n  ✅ Quality report saved: {log_output_dir}/quality_report.log")
    except Exception as e:
        print(f"\n  ⚠️  Could not save quality report: {e}")

    # ═════════════════════════════════════════════════════════════
    # PHASE 8: CLEANUP — DROP ALL TEST CATALOGS EXCEPT BASE
    # ═════════════════════════════════════════════════════════════

    _banner("PHASE 8: CLEANUP — DROPPING TEST CATALOGS")

    try:
        all_catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    except Exception:
        all_catalogs = []

    base_lower = w_catalog.lower()
    _runner_san = _sanitize_name_for_runner(w_business_name)
    runner_ecm_cat = f"{_runner_san}_ecm_v1".lower()
    runner_mvm_cat = f"{_runner_san}_mvm_v1".lower()
    runner_temp_cat = f"{_runner_san}_temp".lower()
    catalogs_to_drop = []
    for cat in all_catalogs:
        cl = cat.lower()
        if cl == base_lower:
            continue
        if cl.startswith(base_lower + "_"):
            catalogs_to_drop.append(cat)
            continue
        if cl in (runner_ecm_cat, runner_mvm_cat, runner_temp_cat):
            catalogs_to_drop.append(cat)
            continue
        for tr in R:
            cp = (tr.params.get("catalog_prefix") or "").lower()
            cs = (tr.params.get("catalog_suffix") or "").lower()
            if cp and cl.startswith(cp):
                catalogs_to_drop.append(cat)
                break
            if cs and not cp and cl.endswith(cs):
                catalogs_to_drop.append(cat)
                break

    catalogs_to_drop = sorted(set(catalogs_to_drop))
    if catalogs_to_drop:
        print(f"  Found {len(catalogs_to_drop)} test catalog(s) to drop (keeping `{w_catalog}`):")
        for cat in catalogs_to_drop:
            try:
                spark.sql(f"DROP CATALOG IF EXISTS `{cat}` CASCADE")
                print(f"    ✅ Dropped `{cat}`")
            except Exception as e:
                print(f"    ⚠️  Failed to drop `{cat}`: {e}")
    else:
        print(f"  No extra test catalogs found to clean up (only `{w_catalog}` exists)")

    print(f"  [{_ts()}] Cleanup complete — only `{w_catalog}` remains")

    # ═════════════════════════════════════════════════════════════
    # PHASE 9: FINAL RESULTS
    # ═════════════════════════════════════════════════════════════

    _banner("PHASE 9: FINAL RESULTS")

    print(f"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║  🧪 VIBE TESTER — FINAL RESULTS                                                  ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  Business:     {w_business_name:<63} ║
║  Catalog:      {w_catalog:<63} ║
║  Agent NB:     {agent_notebook[:63]:<63} ║
║  Runner NB:    {runner_notebook[:63]:<63} ║
║  Total Tests:  {len(R):<63} ║
║  ✅ Passed:    {passed:<63} ║
║  ❌ Failed:    {failed:<63} ║
║  ⏭️  Skipped:   {skipped:<63} ║
║  Duration:     {_fmt_duration(total_duration):<63} ║
║  Quality:      {f'{overall_score:.1f}% {sentiment}':<63} ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  📂 Logs:      {log_output_dir[:63]:<63} ║
║    • merged_info.log    — All info+error logs combined per test                  ║
║    • merged_error.log   — All error logs combined per test                       ║
║    • test_summary.log   — Test results and parameters                            ║
║    • quality_report.log — Convention verification & quality audit                 ║
╚══════════════════════════════════════════════════════════════════════════════════╝
""")

    _sub_banner("Per-test Results")
    for tr in R:
        icon = "✅" if tr.status == "PASSED" else ("❌" if tr.status == "FAILED" else "⏭️")
        dur = _fmt_duration(tr.duration_seconds)
        nc = tr.params.get("naming_convention", "-") if tr.params else "-"
        pk = tr.params.get("primary_key_suffix", "-") if tr.params else "-"
        samp = tr.params.get("generate_samples", "-") if tr.params else "-"
        ar = audit_results.get(tr.test_name)
        conv_str = f"conv={ar['score']:.0f}%" if ar and ar.get("score", -1) >= 0 else ""
        print(f"  {icon} {tr.test_label:<50} {tr.status:<8} {dur:>10}  nc={nc} pk={pk} samples={samp} {conv_str}")

    if failed > 0:
        print(f"\n⚠️  {failed} test(s) FAILED. Check merged_error.log for details.")
    elif passed == len(R):
        print(f"\n🎉 All {passed} tests PASSED!")
    else:
        print(f"\n📋 {passed} passed, {skipped} skipped (due to upstream failures).")


In [0]:
# ============================================================================
# UNIT TESTS — Pure function tests for pipeline fixes (no Spark required)
# Run this cell standalone to validate the fixes.
# ============================================================================

import re
import sys
from collections import defaultdict

_unit_test_failures = []
_unit_test_passed = 0

def _assert_eq(test_name, actual, expected):
    global _unit_test_passed
    if actual == expected:
        _unit_test_passed += 1
    else:
        _unit_test_failures.append(f"  FAIL [{test_name}]: expected {expected!r}, got {actual!r}")

def _assert_true(test_name, value):
    global _unit_test_passed
    if value:
        _unit_test_passed += 1
    else:
        _unit_test_failures.append(f"  FAIL [{test_name}]: expected True, got {value!r}")

def _assert_false(test_name, value):
    global _unit_test_passed
    if not value:
        _unit_test_passed += 1
    else:
        _unit_test_failures.append(f"  FAIL [{test_name}]: expected False, got {value!r}")


# ── Test 1: FORBIDDEN_GENERIC_DOMAIN_NAMES constant ────────────────

FORBIDDEN_GENERIC_DOMAIN_NAMES = frozenset({
    'utilities', 'infrastructure', 'services', 'support', 'platform',
    'common', 'core', 'base', 'general', 'misc', 'other', 'admin', 'auxiliary',
    'analytics', 'reporting', 'intelligence', 'insights', 'metrics', 'kpi', 'dashboard',
    'bi', 'datawarehouse', 'datamarts', 'logging', 'etl', 'integration', 'audit_trail',
    'batch_control', 'system', 'technical', 'data', 'info', 'api',
    'technology', 'regulatory',
    'additional', 'miscellaneous', 'extra', 'analysis', 'processing', 'management',
})
SYSTEM_MANAGED_DOMAIN_NAMES = frozenset({'shared'})

_assert_true("forbidden_contains_infrastructure", 'infrastructure' in FORBIDDEN_GENERIC_DOMAIN_NAMES)
_assert_true("forbidden_contains_regulatory", 'regulatory' in FORBIDDEN_GENERIC_DOMAIN_NAMES)
_assert_false("forbidden_not_contains_citizen", 'citizen' in FORBIDDEN_GENERIC_DOMAIN_NAMES)
_assert_false("forbidden_not_contains_finance", 'finance' in FORBIDDEN_GENERIC_DOMAIN_NAMES)
_assert_true("shared_is_system_managed", 'shared' in SYSTEM_MANAGED_DOMAIN_NAMES)
_assert_false("shared_not_in_forbidden", 'shared' in FORBIDDEN_GENERIC_DOMAIN_NAMES)


# ── Test 2: classify_pii_subtype ────────────────────────────────────

def classify_pii_subtype(attr_name_lower):
    if any(p in attr_name_lower for p in ('email', 'e_mail', 'mail_address')):
        return 'restricted,pii_email'
    if any(p in attr_name_lower for p in ('phone', 'mobile', 'cell_number', 'telephone', 'fax_number', 'msisdn', 'contact_number')):
        return 'restricted,pii_phone'
    if any(p in attr_name_lower for p in ('address', 'street', 'postal_code', 'zip_code', 'po_box', 'geolocation', 'latitude', 'longitude', 'gps_coordinate')):
        return 'restricted,pii_address'
    if any(p in attr_name_lower for p in ('credit_card', 'card_number', 'pan_number', 'debit_card', 'card_pan', 'bank_account', 'iban', 'account_number', 'routing_number', 'swift_code', 'cvv', 'pin_code', 'card_pin', 'salary', 'wage', 'compensation', 'net_pay', 'gross_pay')):
        return 'restricted,pii_financial'
    if any(p in attr_name_lower for p in ('medical_record', 'diagnosis', 'prescription', 'health_condition', 'disability', 'medical_treatment', 'blood_type')):
        return 'restricted,pii_health'
    if any(p in attr_name_lower for p in ('biometric', 'fingerprint', 'face_id', 'retina')):
        return 'restricted,pii_biometric'
    if any(p in attr_name_lower for p in ('national_id', 'ssn', 'social_security', 'emirates_id', 'civil_id', 'identity_number', 'eid_number', 'passport', 'tax_id', 'tin_number', 'vat_id', 'driver_license', 'licence_number', 'ip_address', 'mac_address', 'imei', 'imsi', 'iccid')):
        return 'restricted,pii_identifier'
    return 'restricted,pii_identifier'

_assert_eq("pii_email", classify_pii_subtype("customer_email"), "restricted,pii_email")
_assert_eq("pii_phone", classify_pii_subtype("mobile_number"), "restricted,pii_phone")
_assert_eq("pii_address", classify_pii_subtype("home_address"), "restricted,pii_address")
_assert_eq("pii_financial_salary", classify_pii_subtype("base_salary"), "restricted,pii_financial")
_assert_eq("pii_financial_iban", classify_pii_subtype("iban_code"), "restricted,pii_financial")
_assert_eq("pii_health", classify_pii_subtype("medical_record_id"), "restricted,pii_health")
_assert_eq("pii_biometric", classify_pii_subtype("fingerprint_hash"), "restricted,pii_biometric")
_assert_eq("pii_identifier_ssn", classify_pii_subtype("ssn_number"), "restricted,pii_identifier")
_assert_eq("pii_identifier_passport", classify_pii_subtype("passport_number"), "restricted,pii_identifier")
_assert_eq("pii_geolocation", classify_pii_subtype("gps_coordinate"), "restricted,pii_address")
_assert_eq("pii_latitude", classify_pii_subtype("latitude"), "restricted,pii_address")


# ── Test 3: PII regex matching ──────────────────────────────────────

PII_CANDIDATE_RE = re.compile(
    r'(?:^|_)('
    r'email|e_mail|mail_address'
    r'|first_name|last_name|middle_name|full_name|given_name|family_name|surname'
    r'|customer_name|subscriber_name|employee_name|patient_name|user_name|person_name'
    r'|applicant_name|candidate_name|recipient_name|claimant_name|petitioner_name'
    r'|resident_name|tenant_name|participant_name|witness_name|dependent_name'
    r'|spouse_name|complainant_name|next_of_kin|emergency_contact_name|member_name'
    r'|insured_name|contractor_name|vendor_contact_name|driver_name|operator_name'
    r'|contact_name|caller_name|account_holder_name|beneficiary_name|guardian_name'
    r'|phone|mobile|cell_number|telephone|fax_number|msisdn|contact_number'
    r'|dob|date_of_birth|birth_date|birthday'
    r'|home_address|mailing_address|street_address|residential_address|postal_address'
    r'|address|street|postal_code|zip_code|po_box'
    r'|national_id|ssn|social_security|emirates_id|civil_id|identity_number|eid_number'
    r'|passport|passport_number'
    r'|credit_card|card_number|pan_number|debit_card|card_pan'
    r'|bank_account|iban|account_number|routing_number|swift_code'
    r'|cvv|pin_code|card_pin'
    r'|ip_address|mac_address'
    r'|imei|imsi|iccid'
    r'|biometric|fingerprint|face_id|retina'
    r'|salary|wage|compensation|net_pay|gross_pay'
    r'|medical_record|diagnosis|prescription|health_condition|disability'
    r'|medical_treatment|blood_type'
    r'|tax_id|tin_number|vat_id'
    r'|driver_license|licence_number'
    r'|geolocation|latitude|longitude|gps_coordinate'
    r')(?:_|$)',
    re.IGNORECASE
)
PII_FALSE_POSITIVE_RE = re.compile(
    r'(?:^|_)('
    r'equipment_serial|sensor_serial|chip_serial|device_serial|asset_serial'
    r'|surface_treatment|pavement_treatment|road_treatment|last_treatment'
    r'|treatment_type|treatment_date|treatment_plan'
    r'|claim_filed|claim_status'
    r'|account_number_id|account_number_type'
    r'|latitude_range|longitude_range'
    r'|address_type|address_format|address_count|email_type|email_format'
    r')(?:_|$)',
    re.IGNORECASE
)

_assert_true("pii_re_matches_email", bool(PII_CANDIDATE_RE.search("customer_email")))
_assert_true("pii_re_matches_ssn", bool(PII_CANDIDATE_RE.search("ssn_number")))
_assert_true("pii_re_matches_latitude", bool(PII_CANDIDATE_RE.search("latitude")))
_assert_true("pii_re_matches_compensation", bool(PII_CANDIDATE_RE.search("compensation_amount")))
_assert_false("pii_re_not_match_status", bool(PII_CANDIDATE_RE.search("claim_status")))
_assert_false("pii_re_not_match_record_id", bool(PII_CANDIDATE_RE.search("record_id")))
_assert_true("pii_fp_catches_equipment_serial", bool(PII_FALSE_POSITIVE_RE.search("equipment_serial")))
_assert_true("pii_fp_catches_address_type", bool(PII_FALSE_POSITIVE_RE.search("address_type")))
_assert_true("pii_fp_catches_email_format", bool(PII_FALSE_POSITIVE_RE.search("email_format")))
_assert_false("pii_fp_not_catch_real_email", bool(PII_FALSE_POSITIVE_RE.search("customer_email")))
_assert_true("pii_fp_catches_latitude_range", bool(PII_FALSE_POSITIVE_RE.search("latitude_range")))


# ── Test 4: Cross-domain relocation threshold scaling ───────────────

def _compute_xd_threshold(total_non_shared_domains):
    return max(3, total_non_shared_domains // 3)

def _compute_xd_max_relocations(total_products):
    return max(5, total_products // 7)

_assert_eq("xd_threshold_3_domains", _compute_xd_threshold(3), 3)
_assert_eq("xd_threshold_9_domains", _compute_xd_threshold(9), 3)
_assert_eq("xd_threshold_12_domains", _compute_xd_threshold(12), 4)
_assert_eq("xd_threshold_15_domains", _compute_xd_threshold(15), 5)
_assert_eq("xd_threshold_21_domains", _compute_xd_threshold(21), 7)
_assert_eq("xd_max_reloc_20_products", _compute_xd_max_relocations(20), 5)
_assert_eq("xd_max_reloc_70_products", _compute_xd_max_relocations(70), 10)
_assert_eq("xd_max_reloc_209_products", _compute_xd_max_relocations(209), 29)


# ── Test 5: Bidirectional FK detection ──────────────────────────────

def _detect_bidir_fks(attributes_data):
    fk_index = defaultdict(list)
    for a in attributes_data:
        fk = (a.get('foreign_key_to') or '').strip()
        if not fk or '.' not in fk:
            continue
        parts = fk.split('.')
        if len(parts) < 2:
            continue
        src = f"{(a.get('domain') or '').lower()}.{(a.get('product') or '').lower()}"
        tgt = f"{parts[0].lower()}.{parts[1].lower()}"
        if src != tgt:
            fk_index[(src, tgt)].append(a)
    bidir_pairs = []
    checked = set()
    for (s, t) in fk_index:
        if (s, t) in checked:
            continue
        if (t, s) in fk_index:
            checked.add((s, t))
            checked.add((t, s))
            bidir_pairs.append((s, t))
    return bidir_pairs

attrs_bidir = [
    {'domain': 'billing', 'product': 'account', 'attribute': 'profile_id', 'foreign_key_to': 'customer.profile.profile_id'},
    {'domain': 'customer', 'product': 'profile', 'attribute': 'billing_account_id', 'foreign_key_to': 'billing.account.account_id'},
    {'domain': 'order', 'product': 'order', 'attribute': 'customer_id', 'foreign_key_to': 'customer.profile.profile_id'},
]
attrs_no_bidir = [
    {'domain': 'order', 'product': 'order', 'attribute': 'customer_id', 'foreign_key_to': 'customer.profile.profile_id'},
    {'domain': 'order', 'product': 'line_item', 'attribute': 'order_id', 'foreign_key_to': 'order.order.order_id'},
]
_assert_eq("bidir_detected_count", len(_detect_bidir_fks(attrs_bidir)), 1)
_assert_eq("bidir_detected_pair", _detect_bidir_fks(attrs_bidir)[0], ('billing.account', 'customer.profile'))
_assert_eq("no_bidir_clean", len(_detect_bidir_fks(attrs_no_bidir)), 0)


# ── Test 6: Attribute importance scoring ────────────────────────────

_HIGH_VALUE_SUFFIXES = {'_id', '_date', '_at', '_code', '_type', '_status', '_name', '_key', '_number'}

def _attr_importance(a):
    if not isinstance(a, dict): return -1
    if a.get('is_primary_key'): return 1000
    if a.get('foreign_key_to'): return 900
    an = a.get('attribute', '').lower()
    if a.get('is_nullable') == False or a.get('nullable') == False: return 700
    if a.get('tags') and ('pii' in a.get('tags', '').lower() or 'restricted' in a.get('tags', '').lower()): return 600
    for sfx in _HIGH_VALUE_SUFFIXES:
        if an.endswith(sfx): return 500
    if an in ('created_at', 'updated_at', 'created_date', 'modified_date', 'is_active', 'status', 'name', 'description', 'version'): return 500
    return 100

_assert_eq("importance_pk", _attr_importance({'is_primary_key': True, 'attribute': 'id'}), 1000)
_assert_eq("importance_fk", _attr_importance({'foreign_key_to': 'a.b.c', 'attribute': 'ref_id'}), 900)
_assert_eq("importance_not_null", _attr_importance({'attribute': 'foo', 'is_nullable': False}), 700)
_assert_eq("importance_pii", _attr_importance({'attribute': 'email', 'tags': 'restricted,pii_email'}), 600)
_assert_eq("importance_status_suffix", _attr_importance({'attribute': 'claim_status'}), 500)
_assert_eq("importance_regular", _attr_importance({'attribute': 'notes'}), 100)


# ── Test 7: Domain fallback — substring match min length ────────────

def _would_substring_match(orig, candidate):
    return len(orig) >= 4 and len(candidate) >= 4 and (orig.lower() in candidate or candidate in orig.lower())

_assert_false("short_no_match_hr_in_shared", _would_substring_match("hr", "shared"))
_assert_true("long_match_workforce_in_workforcemgmt", _would_substring_match("workforce", "workforcemgmt"))
_assert_false("no_false_match_rev_in_revenue", _would_substring_match("rev", "revenue"))
_assert_true("match_infra_in_infrastructure", _would_substring_match("infra", "infrastructure"))


# ── Test 8: Stale thread reclaim logic ──────────────────────────────

def _reclaim_stale(current_count):
    return max(0, current_count - (current_count // 3))

_assert_eq("reclaim_from_0", _reclaim_stale(0), 0)
_assert_eq("reclaim_from_3", _reclaim_stale(3), 2)
_assert_eq("reclaim_from_15", _reclaim_stale(15), 10)
_assert_eq("reclaim_from_20", _reclaim_stale(20), 14)


# ── SUMMARY ─────────────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"UNIT TEST RESULTS: {_unit_test_passed} passed, {len(_unit_test_failures)} failed")
print(f"{'='*60}")
if _unit_test_failures:
    for f in _unit_test_failures:
        print(f)
    raise AssertionError(f"{len(_unit_test_failures)} unit test(s) failed")
else:
    print("All unit tests PASSED")

In [0]:
# ============================================================================
# RUN MAIN
# ============================================================================

if __name__ == "__main__":
    jobs_result = main()
    #create_widgets()